## SNN + DeepMimic + APEX DecAP + Multi-Critic PPO + Terrain Curriculum（安定化/1024環境版）

この版では、APEX標準45次元Actor観測を維持しつつ、APEX公式実装との比較で見つかった差分を修正しています。

主な変更: 1024環境、command range `vx=±0.6 / vy=±0.6 / wz=±1.0`、APEX準拠PD (`Kp=20, Kd=0.5`) と action scale 0.25、torque limit、77D privileged critic、quaternion imitation、impact reduction、reference index同期、環境数補正DecAP、Adaptive-KL PPO、NaN/Inf検出、SNN sequence minibatch。


In [1]:
# !pip install -U pip
# !pip install numpy scipy gymnasium tensorboard torch "imageio[ffmpeg]" pillow
# !pip install mujoco mujoco-warp warp-lang


## 2. インポートと実行環境の判定

ライブラリを読み込む。Google Colab で開かれた場合は Drive をマウントし、依存パッケージを自動インストールする。


In [2]:
import os
# ヘッドレス環境でのオフスクリーン描画 (mujoco.Renderer) に EGL を使う
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import sys
ENV_COLAB = "google.colab" in sys.modules
if ENV_COLAB:
    from google.colab import drive, runtime
    drive.mount('/content/drive')
    %cd /content/drive/My Drive/Colab Notebooks
    !pip install scipy gymnasium tensorboard mujoco mujoco-warp warp-lang "imageio[ffmpeg]" pillow

import atexit
from copy import deepcopy
from datetime import datetime
from importlib import metadata
import itertools
import shutil
import tempfile
import time
import traceback
import warnings

import imageio
import numpy as np
from PIL import Image, ImageDraw, ImageFont

import mujoco
import mujoco_warp as mjw
import warp as wp
import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

wp.config.log_level = 30          # warp の起動バナーを抑制
wp.init()

mjw_ver = getattr(mjw, "__version__", None) or metadata.version("mujoco-warp")
print(f"mujoco {mujoco.__version__} / mujoco_warp {mjw_ver} / warp {wp.__version__} "
      f"/ torch {torch.__version__}")


mujoco 3.13.0 / mujoco_warp 3.13.0 / warp 1.17.0 / torch 2.5.1+cu121


## 3. 参照モーション生成器(逆運動学トロット)

学習のお手本になる関節角軌道をつくるセル。対角2脚ずつ足を運ぶトロットの足先軌道を計画し、脚の逆運動学で 12 関節の角度テーブル(1周期分)に変換する。

このセルに書かれているもの:

- `leg_fk()` — 関節角 → 足先位置の順運動学(逆運動学の検証用)
- `leg_ik()` — 足先位置 → 3関節角の解析逆運動学
- `standing_pose()` — 指定した胴体高さで立つ立位関節角
- `OmniTrot` — 指令速度 (vx, vy, ωz) からトロット1周期の関節角テーブルを作る参照モーション生成器
- `check_limits()` — 参照モーションが関節可動域・IK 到達範囲に収まるかの一括チェック


In [3]:
"""Go2 の全方向トロット参照モーション生成器。

速度指令 (vx, vy, ωz) に対し、足の接地位置を決めてから逆運動学で関節角を求める。

- 前進/後退/横移動/旋回とその組み合わせを同じ枠組みで生成する
- 横移動・旋回には外転(hip)関節が要るため、IK は 3 自由度の解析解
- 接地脚の足先はワールド座標に固定される (構造的に滑らない)
"""
import numpy as np

# ---- go2.xml から読み取った寸法 ----
L1     = 0.213                                # thigh リンク長
L2     = float(np.hypot(0.002, 0.213))        # calf 等価リンク長 (足geomは calf 座標系で (-0.002, 0, -0.213))
DELTA  = float(np.arctan2(0.002, 0.213))      # calf リンクの前後オフセット角
D_HIP  = 0.0955                               # 外転ピボット→thigh関節の横オフセット
FOOT_R = 0.022                                # 足geom半径 = 接地時の足中心高さ

ABD_RANGE  = (-1.0472, 1.0472)
CALF_RANGE = (-2.7227, -0.83776)

# (名前, hip_x, hip_y, 左右符号s, 位相オフセット, thigh可動域)
# トロット: 対角ペアが同位相 FL,RR = 0.0 / FR,RL = 0.5
LEGS = [
    ("FL",  0.1934,  0.0465, +1, 0.0, (-1.5708, 3.4907)),
    ("FR",  0.1934, -0.0465, -1, 0.5, (-1.5708, 3.4907)),
    ("RL", -0.1934,  0.0465, +1, 0.5, (-0.5236, 4.5379)),
    ("RR", -0.1934, -0.0465, -1, 0.0, (-0.5236, 4.5379)),
]
QPOS_ORDER = ["FL", "FR", "RL", "RR"]         # go2.xml の qpos[7:] の脚の並び


# ---------------------------------------------------------------- 脚のFK/IK

def _rx(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def _ry(t):
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def leg_fk(angles, s):
    """(θ_abd, θ_thigh, θ_calf) → 外転ピボット基準の足中心位置 (x,y,z)。"""
    t1, t2, t3 = angles
    p = (np.array([0.0, s * D_HIP, 0.0])
         + _ry(t2) @ np.array([0.0, 0.0, -L1])
         + _ry(t2 + t3) @ np.array([-0.002, 0.0, -0.213]))
    return _rx(t1) @ p


def leg_ik(p, s):
    """外転ピボット基準の足先目標 p=(x,y,z) → (θ_abd, θ_thigh, θ_calf)。解析解。

    1) 外転角: 脚の矢状面は外転軸から距離 |D_HIP| の平面にあるので、
       y-z 平面で cosθ1·py + sinθ1·pz = s·D_HIP を解く (足が下にある分岐を選択)。
    2) 残りは矢状面内の 2 リンク IK (余弦定理)。膝の可動域が負なので分岐は一意。
    """
    px, py, pz = p
    d = s * D_HIP
    r = np.hypot(py, pz)
    th1 = np.arctan2(pz, py) + np.arccos(np.clip(d / r, -1.0, 1.0))
    qz = -np.sqrt(max(r * r - d * d, 1e-12))          # 矢状面内での足の深さ
    c3 = np.clip((px * px + qz * qz - L1 * L1 - L2 * L2) / (2 * L1 * L2), -1.0, 1.0)
    ph3 = -np.arccos(c3)                               # 膝は常に負方向(後方)に畳む
    th2 = np.arctan2(-px, -qz) - np.arctan2(L2 * np.sin(ph3), L1 + L2 * np.cos(ph3))
    th3 = ph3 - DELTA
    return np.array([th1, th2, th3])


def standing_pose(h0=0.33):
    """高さ h0 で足を公称接地点に置いた立位の関節角12個 (qpos[7:] の並び)。"""
    q = np.zeros(12)
    for i, (name, hx, hy, s, off, trng) in enumerate(LEGS):
        local = np.array([0.0, s * D_HIP, FOOT_R - h0])
        k = QPOS_ORDER.index(name)
        q[3 * k:3 * k + 3] = leg_ik(local, s)
    return q


# ---------------------------------------------------------------- 生成器

class OmniTrot:
    """速度指令 (vx, vy, ωz) から全方向トロットの参照モーションを生成する。

    vx, vy : 胴体座標系の前後・左右速度 [m/s]
    wz     : ヨー角速度 [rad/s]
    T      : ストライド周期 [s] / beta: デューティ比 / h0: 胴体高さ [m]
    h_swing: 遊脚の持ち上げ高さ [m]

    接地脚の足先(foothold)はワールド座標に固定する。foothold は「接地中間時刻での
    公称接地点の位置」に置く (Raibert 風)。
    """

    def __init__(self, vx=0.0, vy=0.0, wz=0.0,
                 T=0.4, beta=0.5, h0=0.33, h_swing=0.05):
        self.vx, self.vy, self.wz = float(vx), float(vy), float(wz)
        self.T, self.beta, self.h0, self.h_swing = T, beta, h0, h_swing

    # -- 胴体のワールド軌道 (等速の並進+旋回 → 円弧) --
    def base_pose(self, t):
        """時刻 t の胴体位置 (x, y) とヨー角 ψ。"""
        w = self.wz
        ps = w * t
        if abs(w) < 1e-9:
            return self.vx * t, self.vy * t, ps
        x = (self.vx * np.sin(ps) + self.vy * (np.cos(ps) - 1.0)) / w
        y = (self.vx * (1.0 - np.cos(ps)) + self.vy * np.sin(ps)) / w
        return x, y, ps

    # -- foothold: 第nストライドの接地点 (ワールドxy) --
    def _foothold(self, i, n):
        name, hx, hy, s, off, trng = LEGS[i]
        t_mid = (n - off + 0.5 * self.beta) * self.T   # 接地区間の中間時刻
        x, y, ps = self.base_pose(t_mid)
        c, sn = np.cos(ps), np.sin(ps)
        nx, ny = hx, hy + s * D_HIP                    # 公称接地点(胴体座標)
        return np.array([x + c * nx - sn * ny, y + sn * nx + c * ny])

    # -- 足先のワールド位置 --
    def foot_world(self, i, t):
        """時刻 t の脚 i の足中心ワールド位置。接地中は foothold に固定し、遊脚中は次の foothold へ補間する。"""
        off = LEGS[i][4]
        u = t / self.T + off
        n = int(np.floor(u))
        ph = u - n
        if ph < self.beta:                             # 接地: ワールドに固定
            f = self._foothold(i, n)
            return np.array([f[0], f[1], FOOT_R])
        sw = (ph - self.beta) / (1.0 - self.beta)      # 遊脚: 次のfootholdへ
        f0, f1 = self._foothold(i, n), self._foothold(i, n + 1)
        xy = f0 + (f1 - f0) * 0.5 * (1.0 - np.cos(np.pi * sw))
        z = FOOT_R + self.h_swing * np.sin(np.pi * sw)
        return np.array([xy[0], xy[1], z])

    # -- 参照 qpos (19) --
    def qpos_at(self, t):
        """時刻 t の参照 qpos (19): 胴体位置3 + 姿勢クォータニオン4 + 関節12。"""
        x, y, ps = self.base_pose(t)
        c, sn = np.cos(ps), np.sin(ps)
        q = np.zeros(19)
        q[0:3] = [x, y, self.h0]
        q[3], q[6] = np.cos(ps / 2), np.sin(ps / 2)    # ヨーのみのクォータニオン
        for i, (name, hx, hy, s, off, trng) in enumerate(LEGS):
            hipw = np.array([x + c * hx - sn * hy, y + sn * hx + c * hy, self.h0])
            d = self.foot_world(i, t) - hipw
            local = np.array([c * d[0] + sn * d[1],    # ワールド→胴体(ヨー)座標
                              -sn * d[0] + c * d[1], d[2]])
            k = QPOS_ORDER.index(name)
            q[7 + 3 * k:10 + 3 * k] = leg_ik(local, s)
        return q

    # -- 参照 qvel (18): ベースは解析、関節は有限差分 --
    def qvel_at(self, t, eps=1e-4):
        ps = self.wz * t
        c, sn = np.cos(ps), np.sin(ps)
        v = np.zeros(18)
        v[0] = c * self.vx - sn * self.vy              # ワールド系の線速度
        v[1] = sn * self.vx + c * self.vy
        v[5] = self.wz                                 # ヨーのみなので局所系でも同じ
        q0, q1 = self.qpos_at(t - eps), self.qpos_at(t + eps)
        v[6:] = (q1[7:] - q0[7:]) / (2 * eps)
        return v

    # -- 1周期ぶんの関節参照 (模倣学習の報酬テーブル用) --
    def joint_cycle(self, fps=50):
        """1ストライド周期の関節角・関節速度テーブル (n,12)×2。T*fps は整数にすること。"""
        n = int(round(self.T * fps))
        q = np.array([self.qpos_at(k / fps)[7:] for k in range(n)])
        dq = np.array([self.qvel_at(k / fps)[6:] for k in range(n)])
        return q, dq

    # -- 任意長のモーション --
    def qpos_motion(self, duration, fps=50):
        n = int(round(duration * fps))
        return np.array([self.qpos_at(k / fps) for k in range(n)])


def check_limits(motion):
    """モーション(…,19)の12関節が可動域内かを返す (ok, 詳細リスト)。"""
    rngs = []
    for k, name in enumerate(QPOS_ORDER):
        i = [l[0] for l in LEGS].index(name)
        trng = LEGS[i][5]
        rngs += [ABD_RANGE, trng, CALF_RANGE]
    ok, rows = True, []
    q = motion.reshape(-1, motion.shape[-1])
    for j, (lo, hi) in enumerate(rngs):
        mn, mx = q[:, 7 + j].min(), q[:, 7 + j].max()
        inside = (lo - 1e-9 <= mn) and (mx <= hi + 1e-9)
        ok &= inside
        rows.append((mn, mx, lo, hi, inside))
    return ok, rows


## 4. ハイパーパラメータ — SNN + DeepMimic + DecAP + Multi-Critic PPO


In [4]:
# ============================================================
# ユーザー設定：通常はここだけ変更すればよい
# ============================================================
RUN_NAME = "go2_snn_apex_rough_fixed1024"      # ← 保存ファイル名を自由に変更
RUN_DATE = datetime.now().strftime("%m-%d")

BASE_SCENE_XML = "scene_flat.xml"     # 元のGo2平地MJCF
USE_APEX_ROUGH_TERRAIN = True          # True: APEX型不整地curriculum
TEST_ON_ROUGH_TERRAIN = False          # 通常評価はFalse（平地・Prior OFF）
TERRAIN_SEED = 39

# APEX論文/公式コードに合わせたterrain atlas
TERRAIN_HORIZONTAL_SCALE = 0.10        # [m]
TERRAIN_VERTICAL_SCALE = 0.005         # [m]
TERRAIN_LENGTH = 8.0                   # 1 tileのx長 [m]
TERRAIN_WIDTH = 8.0                    # 1 tileのy幅 [m]
TERRAIN_ROWS = 10                      # difficulty levels
TERRAIN_COLS = 20                      # terrain-type columns
TERRAIN_MAX_INIT_LEVEL = 5
TERRAIN_PROPORTIONS = (0.2, 0.2, 0.2, 0.2, 0.2)
# APEX公式と同じ25mの外周border。XMLを小さくしたい場合のみ0.0へ変更可能。
TERRAIN_BORDER_SIZE = 25.0
TERRAIN_MEASURE_HEIGHTS_FOR_CRITIC = True   # Actorはblind、Criticのみ高さを使用

# ---- 歩容と指令レンジ ----
T_GAIT, H0, H_SWING = 0.40, 0.33, 0.05
CMD_SCALE = np.array([0.60, 0.60, 1.00], dtype=np.float64)  # vx, vy, wz の学習範囲
CMD_RESAMPLE_TIME = 5.0  # APEX公式と同じ5秒ごとに指令を再サンプル
CYC = int(round(T_GAIT / 0.02))

# ---- サーボ(位置)制御 ----
SERVO_KP, SERVO_KD = 20.0, 0.5
ACT_SCALE = np.full(12, 0.25, dtype=np.float64)
ACTION_CLIP = 100.0  # APEX normalization.clip_actions=100 と同じ。torque側で安全に制限する

# ---- 環境 ----
NUM_ENVS = 1024
PHYSICS_PRESET = "balanced"
USE_CUDA_GRAPH = True
VIDEO_PANEL = (320, 240) if ENV_COLAB else (640, 480)

# ---- APEX / DeepMimic ----
USE_RSI = False
# APEX標準Actorはreference/phaseを入力しない45次元。
ACTOR_USE_PHASE = False

# ---- APEX reward (current official repoに寄せた重み) ----
# 各scaleはAPEX本家と同様、最終的にcontrol dtを掛けて1step報酬へ変換する。
APEX_TRACKING_SIGMA = 0.25
APEX_SIGMA_IMIT_ANGLES = 0.01
APEX_SIGMA_IMIT_FOOT = 0.01
APEX_SIGMA_IMIT_QUAT = 0.5
W_IMIT_ANGLES = 3.5
W_IMIT_FOOT = 2.5
W_IMIT_QUAT = 0.5
W_TRACK_LIN = 2.0
W_TRACK_ANG = 1.5
W_TORQUE = 1.0e-5
W_DOF_ACC = 2.5e-7
W_COLLISION = 1.0
W_ACTION_RATE = 0.01
W_FEET_SLIP = 0.04
W_IMPACT_REDUCTION = 2.5e-3
W_STUMBLE_ROUGH = 0.20
W_ANG_VEL_XY_ROUGH = 0.02
W_ANG_VEL_XY_FLAT = 0.05

# Decaying Action Prior: c_n = gamma^(control_step / k)
# 公式APEXは4096 env / k=100。1024 envでは同じtransition量をPrior付きで経験できるようkを4倍する。
APEX_GAMMA = 0.99
APEX_K_BASE = 100.0
APEX_REFERENCE_ENVS = 4096
APEX_PRIOR_MIN = 0.0

# Multi-Critic advantage weights: [imitation/style, task]
CRITIC_WEIGHTS = (0.5, 0.5)
NUM_CRITICS = 2

# ---- SNN ----
SNN_COMPILE = False
PPO_DETERMINISTIC_POP_ENCODING = True

# ---- PPO ----
SMOKE_TEST = os.environ.get("GO2_SMOKE", "0") == "1"
gpu_id = 0
if gpu_id >= torch.cuda.device_count():
    gpu_id = 0

if SMOKE_TEST:
    NUM_ENVS = 64
    total_iters = 10
    num_steps_per_env = 8
    ppo_epochs = 2
    num_mini_batches = 2
    eval_interval = 5
    save_interval = 10
    max_ep_len = 100
    encoder_pop_dim = decoder_pop_dim = 32
    hidden = (128, 128)
else:
    total_iters = 2_000
    num_steps_per_env = 32
    ppo_epochs = 5
    num_mini_batches = 4
    eval_interval = 100
    save_interval = 500
    max_ep_len = 500
    encoder_pop_dim, decoder_pop_dim = 64, 256
    hidden = (256, 256)

APEX_K = APEX_K_BASE * (APEX_REFERENCE_ENVS / float(NUM_ENVS))

PPO_GAMMA = 0.99
PPO_LAMBDA = 0.95
PPO_CLIP = 0.2
PPO_LR = 3e-4
PPO_VALUE_COEF = 1.0
PPO_ENTROPY_COEF = 0.01
PPO_MAX_GRAD_NORM = 1.0
PPO_INIT_STD = 0.30
PPO_MIN_STD = 0.05
PPO_MAX_STD = 1.00
PPO_DESIRED_KL = 0.01
PPO_LR_MIN = 1.0e-5
PPO_LR_MAX = 1.0e-3
PPO_LOG_RATIO_CLIP = 20.0  # exp() overflow防止。通常はAdaptive KLによりここへ到達しない
PPO_SNN_SEQUENCE_MINIBATCH = True
NORM_CLIP_LIMIT = 50.0

# ============================================================
# 保存先：要求どおり params_日付_APEX / videos_日付_APEX
# RUN_NAME はファイル名のprefixとして使う
# ============================================================
model_dir = f"./params_{RUN_DATE}_APEX3"
video_dir = f"./videos_{RUN_DATE}_APEX3"
terrain_dir = f"./terrains_{RUN_DATE}_APEX3"
logdir = f"runs/{RUN_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{gpu_id}"
for d in (model_dir, video_dir, terrain_dir):
    os.makedirs(d, exist_ok=True)

if torch.cuda.is_available():
    device = torch.device("cuda", index=gpu_id)
else:
    raise RuntimeError("MuJoCo Warp 版は CUDA GPU が必要です")
torch.cuda.set_device(device)

transitions_per_iter = NUM_ENVS * num_steps_per_env
print(f"RUN_NAME={RUN_NAME}")
print(f"保存先: {model_dir} / {video_dir} / {terrain_dir}")
print(f"使用デバイス: {device} ({torch.cuda.get_device_name(device)})")
print(f"env={NUM_ENVS} / rollout={num_steps_per_env} / PPO iters={total_iters:,}")
print(f"command range: vx=±{CMD_SCALE[0]:.1f}, vy=±{CMD_SCALE[1]:.1f}, wz=±{CMD_SCALE[2]:.1f}")
print(f"PD: Kp={SERVO_KP}, Kd={SERVO_KD}, action_scale={ACT_SCALE[0]:.2f}, DecAP k={APEX_K:.1f}")


RUN_NAME=go2_snn_apex_rough_fixed1024
保存先: ./params_09-24_APEX3 / ./videos_09-24_APEX3 / ./terrains_09-24_APEX3
使用デバイス: cuda:0 (NVIDIA RTX A6000)
env=1024 / rollout=32 / PPO iters=2,000
command range: vx=±0.6, vy=±0.6, wz=±1.0
PD: Kp=20.0, Kd=0.5, action_scale=0.25, DecAP k=400.0


## 5. APEX型Terrain AtlasをMuJoCo XMLとして生成

APEX公式実装は地形をXMLとして保存していません。Pythonでheight fieldを手続き生成し、Isaac Gymでtriangle meshへ変換します。

このNotebookでは同じ考え方をMuJoCoへ移植し、以下を行います。

- 10 difficulty rows × 20 terrain-type columns
- 1 tile = 8 m × 8 m
- smooth slope / rough slope / stairs down / stairs up / discrete obstacles
- MuJoCoの `<hfield elevation="...">` として **XML単体** に保存
- 元の `scene_flat.xml` にhfieldを自動挿入した学習用XMLも生成
- terrain levelは各envごとに保持し、APEXと同じ基準で+1/-1

注意：APEX公式と同じ25 m外周borderを既定で含めています。XMLサイズを抑えたい場合だけ `TERRAIN_BORDER_SIZE=0.0` にできます。


In [5]:

from pathlib import Path
import copy
import xml.etree.ElementTree as ET
from scipy import interpolate

TERRAIN_NAMES = ("smooth_slope", "rough_slope", "stairs_down", "stairs_up", "discrete")


def _pyramid_sloped_raw(n, slope, hscale, vscale, platform_size=3.0):
    """Isaac Gym terrain_utils.pyramid_sloped_terrain と同じ離散化。"""
    h = np.zeros((n, n), dtype=np.int16)
    x = np.arange(n)
    y = np.arange(n)
    cx, cy = n // 2, n // 2
    xx, yy = np.meshgrid(x, y, sparse=True)
    xx = (cx - np.abs(cx - xx)) / max(cx, 1)
    yy = (cy - np.abs(cy - yy)) / max(cy, 1)
    xx = xx.reshape(n, 1)
    yy = yy.reshape(1, n)
    max_height = int(slope * (hscale / vscale) * (n / 2))
    h += (max_height * xx * yy).astype(np.int16)
    p = int(platform_size / hscale / 2)
    x1, x2 = n // 2 - p, n // 2 + p
    y1, y2 = n // 2 - p, n // 2 + p
    min_h = min(int(h[x1, y1]), 0)
    max_h = max(int(h[x1, y1]), 0)
    return np.clip(h, min_h, max_h).astype(np.int16)


def _random_uniform_raw(n, rng, hscale, vscale,
                        min_height=-0.05, max_height=0.05, step=0.005,
                        downsampled_scale=0.2):
    """Isaac Gym terrain_utils.random_uniform_terrain と同じ離散化/補間。"""
    min_h = int(min_height / vscale)
    max_h = int(max_height / vscale)
    step_h = int(step / vscale)
    heights_range = np.arange(min_h, max_h + step_h, step_h)
    down_n = int(n * hscale / downsampled_scale)
    coarse = rng.choice(heights_range, (down_n, down_n))
    x = np.linspace(0, n * hscale, coarse.shape[0])
    y = np.linspace(0, n * hscale, coarse.shape[1])
    f = interpolate.RectBivariateSpline(y, x, coarse)
    xu = np.linspace(0, n * hscale, n)
    yu = np.linspace(0, n * hscale, n)
    return np.rint(f(yu, xu)).astype(np.int16)


def _pyramid_stairs_raw(n, step_width, step_height, hscale, vscale, platform_size=3.0):
    """Isaac Gym terrain_utils.pyramid_stairs_terrain と同じ構造。"""
    h = np.zeros((n, n), dtype=np.int16)
    sw = max(1, int(step_width / hscale))
    sh = int(step_height / vscale)
    platform = int(platform_size / hscale)
    height = 0
    sx, ex, sy, ey = 0, n, 0, n
    while (ex - sx) > platform and (ey - sy) > platform:
        sx += sw; ex -= sw; sy += sw; ey -= sw
        height += sh
        if sx < ex and sy < ey:
            h[sx:ex, sy:ey] = height
    return h


def _discrete_obstacles_raw(n, rng, max_height, hscale, vscale,
                            min_size=1.0, max_size=2.0, num_rects=20,
                            platform_size=3.0):
    """Isaac Gym terrain_utils.discrete_obstacles_terrain と同じ離散候補。"""
    h = np.zeros((n, n), dtype=np.int16)
    mh = int(max_height / vscale)
    min_px = max(1, int(min_size / hscale))
    max_px = max(min_px + 1, int(max_size / hscale))
    platform_px = int(platform_size / hscale)
    heights = np.array([-mh, -mh // 2, mh // 2, mh], dtype=np.int16)
    widths = np.arange(min_px, max_px, 4, dtype=int)
    if widths.size == 0:
        widths = np.array([min_px])
    for _ in range(num_rects):
        w = int(rng.choice(widths)); l = int(rng.choice(widths))
        ii = np.arange(0, max(1, n - w), 4, dtype=int)
        jj = np.arange(0, max(1, n - l), 4, dtype=int)
        si = int(rng.choice(ii)) if ii.size else 0
        sj = int(rng.choice(jj)) if jj.size else 0
        h[si:si+w, sj:sj+l] = int(rng.choice(heights))
    x1 = (n - platform_px) // 2; x2 = (n + platform_px) // 2
    y1 = (n - platform_px) // 2; y2 = (n + platform_px) // 2
    h[x1:x2, y1:y2] = 0
    return h


def _make_apex_tile(row, col, rng):
    n = int(round(TERRAIN_LENGTH / TERRAIN_HORIZONTAL_SCALE))
    difficulty = row / TERRAIN_ROWS
    choice = col / TERRAIN_COLS + 0.001
    slope = difficulty * 0.8
    step_height = 0.05 + 0.11 * difficulty
    obstacle_height = 0.05 + 0.20 * difficulty
    cum = np.cumsum(TERRAIN_PROPORTIONS)

    if choice < cum[0]:
        if choice < cum[0] / 2:
            slope *= -1
        raw = _pyramid_sloped_raw(n, slope, TERRAIN_HORIZONTAL_SCALE, TERRAIN_VERTICAL_SCALE, 3.0)
        kind = "smooth_slope"
    elif choice < cum[1]:
        raw = _pyramid_sloped_raw(n, slope, TERRAIN_HORIZONTAL_SCALE, TERRAIN_VERTICAL_SCALE, 3.0)
        raw = (raw.astype(np.int32) + _random_uniform_raw(
            n, rng, TERRAIN_HORIZONTAL_SCALE, TERRAIN_VERTICAL_SCALE,
            -0.05, 0.05, 0.005, 0.2).astype(np.int32)).clip(-32768, 32767).astype(np.int16)
        kind = "rough_slope"
    elif choice < cum[3]:
        if choice < cum[2]:
            step_height *= -1
            kind = "stairs_down"
        else:
            kind = "stairs_up"
        raw = _pyramid_stairs_raw(n, 0.31, step_height,
                                   TERRAIN_HORIZONTAL_SCALE, TERRAIN_VERTICAL_SCALE, 3.0)
    elif choice < cum[4]:
        raw = _discrete_obstacles_raw(n, rng, obstacle_height,
                                      TERRAIN_HORIZONTAL_SCALE, TERRAIN_VERTICAL_SCALE,
                                      1.0, 2.0, 20, 3.0)
        kind = "discrete"
    else:
        raise RuntimeError("terrain proportions must cover [0,1]")
    return raw, kind


def build_apex_terrain_atlas(seed=TERRAIN_SEED):
    rng = np.random.RandomState(seed)
    nx = int(round(TERRAIN_LENGTH / TERRAIN_HORIZONTAL_SCALE))
    ny = int(round(TERRAIN_WIDTH / TERRAIN_HORIZONTAL_SCALE))
    border = int(round(TERRAIN_BORDER_SIZE / TERRAIN_HORIZONTAL_SCALE))
    atlas = np.zeros((TERRAIN_ROWS * nx + 2 * border,
                      TERRAIN_COLS * ny + 2 * border), dtype=np.int16)
    origins = np.zeros((TERRAIN_ROWS, TERRAIN_COLS, 3), dtype=np.float32)
    kinds = []

    for col in range(TERRAIN_COLS):
        col_kind = None
        for row in range(TERRAIN_ROWS):
            tile, kind = _make_apex_tile(row, col, rng)
            col_kind = kind
            xs = border + row * nx; xe = xs + nx
            ys = border + col * ny; ye = ys + ny
            atlas[xs:xe, ys:ye] = tile
            # APEX: tile centerの2m×2m内の最大高さをspawn origin zにする
            a = int((TERRAIN_LENGTH / 2 - 1.0) / TERRAIN_HORIZONTAL_SCALE)
            b = int((TERRAIN_LENGTH / 2 + 1.0) / TERRAIN_HORIZONTAL_SCALE)
            origin_z = tile[a:b, a:b].max() * TERRAIN_VERTICAL_SCALE
            origins[row, col] = [
                (row + 0.5) * TERRAIN_LENGTH,
                (col + 0.5) * TERRAIN_WIDTH,
                origin_z,
            ]
        kinds.append(col_kind)

    height_m = atlas.astype(np.float32) * TERRAIN_VERTICAL_SCALE
    return {
        "height_raw": atlas,
        "height_m": height_m,
        "origins": origins,
        "column_kind": np.asarray(kinds),
        "horizontal_scale": TERRAIN_HORIZONTAL_SCALE,
        "vertical_scale": TERRAIN_VERTICAL_SCALE,
        "terrain_length": TERRAIN_LENGTH,
        "terrain_width": TERRAIN_WIDTH,
        "rows": TERRAIN_ROWS,
        "cols": TERRAIN_COLS,
        "border_size": TERRAIN_BORDER_SIZE,
        "max_init_level": TERRAIN_MAX_INIT_LEVEL,
    }


def save_terrain_xml(meta, xml_path):
    """MuJoCo hfieldをinline elevationとして保存。PNG等は不要。"""
    h = meta["height_m"]
    hmin, hmax = float(h.min()), float(h.max())
    hrange = max(hmax - hmin, TERRAIN_VERTICAL_SCALE)
    norm = (h - hmin) / hrange

    # MuJoCo XMLはrow orderがmodel内部と上下反転。x/y対応のためtransposeしてflip。
    xml_elev = np.flipud(norm.T).reshape(-1)
    nrow, ncol = norm.T.shape
    total_x = (h.shape[0] - 1) * TERRAIN_HORIZONTAL_SCALE
    total_y = (h.shape[1] - 1) * TERRAIN_HORIZONTAL_SCALE
    # APEXはmeshを-borderだけ平行移動するため、world上のtile originはborderを含まない
    center_x = total_x / 2.0 - float(meta["border_size"])
    center_y = total_y / 2.0 - float(meta["border_size"])
    elev_text = " ".join(f"{v:.5f}" for v in xml_elev)

    root = ET.Element("mujoco", {"model": "apex_terrain_atlas"})
    ET.SubElement(root, "compiler", {"angle": "radian"})
    asset = ET.SubElement(root, "asset")
    ET.SubElement(asset, "hfield", {
        "name": "apex_terrain_hf",
        "nrow": str(nrow), "ncol": str(ncol),
        "size": f"{total_x/2:.6f} {total_y/2:.6f} {hrange:.6f} 1.0",
        "elevation": elev_text,
    })
    world = ET.SubElement(root, "worldbody")
    ET.SubElement(world, "light", {"pos":"0 0 50", "dir":"0 0 -1", "diffuse":"0.8 0.8 0.8"})
    ET.SubElement(world, "geom", {
        "name": "apex_terrain", "type": "hfield", "hfield": "apex_terrain_hf",
        "pos": f"{center_x:.6f} {center_y:.6f} {hmin:.6f}",
        "friction": "1.0 0.005 0.0001", "rgba": "0.55 0.58 0.52 1",
    })
    ET.ElementTree(root).write(xml_path, encoding="utf-8", xml_declaration=True)


def merge_terrain_into_scene(base_xml, terrain_xml, output_xml):
    """元sceneのplaneをhfieldへ置き換えたfull Go2 MJCFを作る。"""
    base_xml = Path(base_xml).resolve()
    terrain_xml = Path(terrain_xml).resolve()
    output_xml = Path(output_xml).resolve()
    tree = ET.parse(base_xml)
    root = tree.getroot()
    troot = ET.parse(terrain_xml).getroot()

    asset = root.find("asset")
    if asset is None:
        asset = ET.Element("asset")
        root.insert(0, asset)
    # 同名terrain assetの重複を除去
    for elem in list(asset):
        if elem.tag == "hfield" and elem.get("name") == "apex_terrain_hf":
            asset.remove(elem)
    hfield = troot.find("./asset/hfield")
    asset.append(copy.deepcopy(hfield))

    world = root.find("worldbody")
    if world is None:
        raise RuntimeError("BASE_SCENE_XML に <worldbody> がありません")
    # worldbody直下のplaneだけを除去。ロボット内部geomは触らない。
    for elem in list(world):
        if elem.tag == "geom" and (elem.get("type") == "plane" or elem.get("name") == "apex_terrain"):
            world.remove(elem)
    terrain_geom = troot.find("./worldbody/geom[@name='apex_terrain']")
    world.insert(0, copy.deepcopy(terrain_geom))

    # include/mesh等の相対パスを壊さないため、base sceneと同じdirectoryへ保存
    output_xml.parent.mkdir(parents=True, exist_ok=True)
    tree.write(output_xml, encoding="utf-8", xml_declaration=True)
    return str(output_xml)


APEX_TERRAIN_META = None
TRAIN_XML = BASE_SCENE_XML
TEST_XML = BASE_SCENE_XML

if USE_APEX_ROUGH_TERRAIN:
    APEX_TERRAIN_META = build_apex_terrain_atlas()
    terrain_only_xml = Path(terrain_dir) / f"{RUN_NAME}_terrain_atlas.xml"
    terrain_meta_npz = Path(terrain_dir) / f"{RUN_NAME}_terrain_meta.npz"
    save_terrain_xml(APEX_TERRAIN_META, terrain_only_xml)
    np.savez_compressed(
        terrain_meta_npz,
        height_raw=APEX_TERRAIN_META["height_raw"],
        origins=APEX_TERRAIN_META["origins"],
        column_kind=APEX_TERRAIN_META["column_kind"],
        horizontal_scale=TERRAIN_HORIZONTAL_SCALE,
        vertical_scale=TERRAIN_VERTICAL_SCALE,
    )

    # 元sceneが存在する実行環境では、Go2込みのtraining XMLを自動生成
    base_path = Path(BASE_SCENE_XML)
    if base_path.exists():
        merged_xml = base_path.resolve().parent / f"{RUN_NAME}_apex_curriculum.xml"
        TRAIN_XML = merge_terrain_into_scene(base_path, terrain_only_xml, merged_xml)
        if TEST_ON_ROUGH_TERRAIN:
            TEST_XML = TRAIN_XML
        print("terrain XML:", terrain_only_xml)
        print("training XML:", TRAIN_XML)
    else:
        print(f"[注意] {BASE_SCENE_XML} がまだ見つかりません。")
        print("scene_flat.xml と同じ場所でこのセルを再実行すると、Go2込みXMLを自動生成します。")
else:
    print("flat terrain mode")


terrain XML: terrains_09-24_APEX3/go2_snn_apex_rough_fixed1024_terrain_atlas.xml
training XML: /workspace/easy-docker-jupyterhub/unitree_mujoco/unitree_robots/go2_2/go2_snn_apex_rough_fixed1024_apex_curriculum.xml


## 5. 参照モーションのバッチ化

OmniTrot と同じ参照モーションを、全ワールド分まとめて GPU 上のテンソルとして計算するセル。

このセルに書かれているもの:

- `BatchTrot` — 全ワールド分の参照モーションを GPU 上のテンソルとしてまとめて計算するバッチ版生成器


In [6]:
_HX  = torch.tensor([l[1] for l in LEGS], dtype=torch.float64)
_HY  = torch.tensor([l[2] for l in LEGS], dtype=torch.float64)
_S   = torch.tensor([float(l[3]) for l in LEGS], dtype=torch.float64)
_OFF = torch.tensor([l[4] for l in LEGS], dtype=torch.float64)
# LEGS の並び (FL,FR,RL,RR) は QPOS_ORDER と同一なので、脚 i がそのまま qpos の i 番目に対応する


class BatchTrot:
    """指令 cmd[N,3] に対する参照モーションをまとめて生成する。

    cmd  : 胴体座標系の (vx, vy, ωz)
    t    : 時刻 [N] または [1] [s]
    h_swing: env ごとの遊脚高さ [N] (指令ゼロのワールドは 0 = 足を持ち上げない)

    入出力はすべて float64。脚の並びは QPOS_ORDER (FL,FR,RL,RR)。
    """

    def __init__(self, device, T=T_GAIT, beta=0.5, h0=H0):
        self.dev = device
        self.T, self.beta, self.h0 = T, beta, h0
        self.hx, self.hy, self.s, self.off = (t.to(device) for t in (_HX, _HY, _S, _OFF))

    # -- 胴体のワールド軌道 --
    def _base_pose(self, cmd, t):
        vx, vy, wz = cmd[..., 0], cmd[..., 1], cmd[..., 2]
        ps = wz * t
        small = wz.abs() < 1e-9
        wzs = torch.where(small, torch.ones_like(wz), wz)          # ゼロ割回避
        x = (vx * torch.sin(ps) + vy * (torch.cos(ps) - 1.0)) / wzs
        y = (vx * (1.0 - torch.cos(ps)) + vy * torch.sin(ps)) / wzs
        return torch.where(small, vx * t, x), torch.where(small, vy * t, y), ps

    # -- foothold: ストライド番号 n[...,4] → ワールド xy [...,4,2] --
    def _foothold(self, cmd, n):
        t_mid = (n - self.off + 0.5 * self.beta) * self.T
        x, y, ps = self._base_pose(cmd.unsqueeze(-2), t_mid)
        c, sn = torch.cos(ps), torch.sin(ps)
        nx, ny = self.hx, self.hy + self.s * D_HIP
        return torch.stack([x + c * nx - sn * ny, y + sn * nx + c * ny], dim=-1)

    # -- 足先のワールド位置 [...,4,3] --
    def _foot_world(self, cmd, t, h_swing):
        u = t.unsqueeze(-1) / self.T + self.off
        n = torch.floor(u)
        ph = u - n
        f0, f1 = self._foothold(cmd, n), self._foothold(cmd, n + 1.0)
        sw = ((ph - self.beta) / (1.0 - self.beta)).unsqueeze(-1)
        xy_sw = f0 + (f1 - f0) * 0.5 * (1.0 - torch.cos(np.pi * sw))
        z_sw = FOOT_R + h_swing.unsqueeze(-1) * torch.sin(np.pi * sw.squeeze(-1))
        stance = ph < self.beta
        xy = torch.where(stance.unsqueeze(-1), f0, xy_sw)          # 接地脚はワールド固定
        z = torch.where(stance, torch.full_like(z_sw, FOOT_R), z_sw)
        return torch.cat([xy, z.unsqueeze(-1)], dim=-1)

    # -- 3自由度 IK の解析解 (leg_ik のテンソル版) --
    @staticmethod
    def _leg_ik(p, s):
        px, py, pz = p[..., 0], p[..., 1], p[..., 2]
        d = s * D_HIP
        r = torch.hypot(py, pz)
        th1 = torch.atan2(pz, py) + torch.acos(torch.clamp(d / r, -1.0, 1.0))
        qz = -torch.sqrt(torch.clamp(r * r - d * d, min=1e-12))
        c3 = torch.clamp((px * px + qz * qz - L1 ** 2 - L2 ** 2) / (2 * L1 * L2), -1.0, 1.0)
        ph3 = -torch.acos(c3)
        th2 = torch.atan2(-px, -qz) - torch.atan2(L2 * torch.sin(ph3), L1 + L2 * torch.cos(ph3))
        return torch.stack([th1, th2, ph3 - DELTA], dim=-1)

    # -- 参照 qpos [N,19] --
    def qpos_at(self, cmd, t, h_swing):
        x, y, ps = self._base_pose(cmd, t)
        c, sn = torch.cos(ps), torch.sin(ps)
        hipw_x = x.unsqueeze(-1) + c.unsqueeze(-1) * self.hx - sn.unsqueeze(-1) * self.hy
        hipw_y = y.unsqueeze(-1) + sn.unsqueeze(-1) * self.hx + c.unsqueeze(-1) * self.hy
        fw = self._foot_world(cmd, t, h_swing)
        dx, dy, dz = fw[..., 0] - hipw_x, fw[..., 1] - hipw_y, fw[..., 2] - self.h0
        cu, su = c.unsqueeze(-1), sn.unsqueeze(-1)
        local = torch.stack([cu * dx + su * dy, -su * dx + cu * dy, dz], dim=-1)   # ワールド→胴体(ヨー)
        ang = self._leg_ik(local, self.s)                                          # [N,4,3]
        q = torch.zeros(cmd.shape[0], 19, dtype=cmd.dtype, device=cmd.device)
        q[:, 0], q[:, 1], q[:, 2] = x, y, self.h0
        q[:, 3], q[:, 6] = torch.cos(ps / 2), torch.sin(ps / 2)
        q[:, 7:] = ang.reshape(-1, 12)
        return q

    # -- 参照 qvel [N,18]: ベースは解析、関節は有限差分 --
    def qvel_at(self, cmd, t, h_swing, eps=1e-4):
        ps = cmd[:, 2] * t
        c, sn = torch.cos(ps), torch.sin(ps)
        v = torch.zeros(cmd.shape[0], 18, dtype=cmd.dtype, device=cmd.device)
        v[:, 0] = c * cmd[:, 0] - sn * cmd[:, 1]
        v[:, 1] = sn * cmd[:, 0] + c * cmd[:, 1]
        v[:, 5] = cmd[:, 2]
        q0 = self.qpos_at(cmd, t - eps, h_swing)
        q1 = self.qpos_at(cmd, t + eps, h_swing)
        v[:, 6:] = (q1[:, 7:] - q0[:, 7:]) / (2 * eps)
        return v

    # -- 1周期ぶんの関節参照テーブル: q_tab[N,CYC,12], dq_tab[N,CYC,12] --
    def joint_cycle(self, cmd, h_swing, fps=50, eps=1e-4):
        """1 ストライド周期の関節参照テーブル q[N,n,12], dq[N,n,12] (n = T*fps)。

        dq は中心差分 (t±eps)。3 本の時刻を 1 バッチにまとめて評価する。
        """
        n = int(round(self.T * fps))
        N = cmd.shape[0]
        t = (torch.arange(n, device=cmd.device, dtype=cmd.dtype) / fps).repeat(N)
        cmd_r = cmd.repeat_interleave(n, dim=0)
        hs_r = h_swing.repeat_interleave(n, dim=0)
        q_all = self.qpos_at(cmd_r.repeat(3, 1), torch.cat([t, t - eps, t + eps]),
                             hs_r.repeat(3))[:, 7:]
        M = N * n
        q = q_all[:M].reshape(N, n, 12)
        dq = ((q_all[2 * M:] - q_all[M:2 * M]) / (2 * eps)).reshape(N, n, 12)
        return q, dq


## 6. バッチ環境(MuJoCo Warp) — APEX 45D Actor + Style/Task reward + Decaying Action Prior

- Actor: 45D (`angular velocity + projected gravity + command + q + dq + last action`)
- Style Critic: joint-angle imitation + foot/end-effector-position imitation
- Task Critic: velocity/yaw tracking + torque / DOF acceleration / collision / action-rate / feet-slip / stumble / angular-velocity regularization
- Actorには足先位置・reference・phaseを入力しません。

In [7]:
# ---- warp kernels ----
@wp.kernel
def _pd_kernel(qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
               target: wp.array2d(dtype=float), ref_target: wp.array2d(dtype=float),
               prior_factor: wp.array(dtype=float), perm: wp.array(dtype=wp.int32),
               torque_limit: wp.array(dtype=float),
               kp: float, kd: float, ctrl: wp.array2d(dtype=float)):
    """APEX position control.

    tau = Kp(q_policy-q) - Kd*dq + c_n*Kp(q_ref-q)
    """
    w, j = wp.tid()
    jj = perm[j]
    q = qpos[w, 7 + jj]
    dq = qvel[w, 6 + jj]
    policy_tau = kp * (target[w, j] - q) - kd * dq
    prior_tau = prior_factor[w] * kp * (ref_target[w, j] - q)
    tau = policy_tau + prior_tau
    lim = torque_limit[j]
    ctrl[w, j] = wp.min(wp.max(tau, -lim), lim)


@wp.kernel
def _write_state_kernel(qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float),
                        qacc_ws: wp.array2d(dtype=float),
                        new_q: wp.array2d(dtype=float), new_v: wp.array2d(dtype=float),
                        idx: wp.array(dtype=wp.int32), nq: int, nv: int):
    """idx で指定したワールドだけ状態を差し替える (auto-reset 用)。"""
    i = wp.tid()
    w = idx[i]
    for k in range(nq):
        qpos[w, k] = new_q[i, k]
    for k in range(nv):
        qvel[w, k] = new_v[i, k]
        qacc_ws[w, k] = 0.0


PHYSICS_PRESETS = {
    "exact": {},
    "balanced": dict(
        solver=mujoco.mjtSolver.mjSOL_NEWTON,
        iterations=8,
        ls_iterations=8,
    ),
    "fast": dict(
        solver=mujoco.mjtSolver.mjSOL_NEWTON,
        iterations=1,
        ls_iterations=4,
        cone=mujoco.mjtCone.mjCONE_PYRAMIDAL,
        impratio=1.0,
    ),
}


class Go2ImitationWarpEnv:
    """MuJoCo-Warp版 SNN + DeepMimic + APEX environment.

    Actor observation (45D, APEX標準):
      base angular velocity(3), projected gravity(3), command(3),
      q-default(12), dq(12), last/executed action(12)

    reward group 0 (Style): joint-angle imitation + foot-position imitation
    reward group 1 (Task): velocity tracking + APEX regularization
    """

    PERM = np.array([3, 4, 5, 0, 1, 2, 9, 10, 11, 6, 7, 8])
    REWARD_KEYS = (
        "reward/imitation_angles", "reward/imitation_foot", "reward/imitation_quat",
        "reward/tracking_lin", "reward/tracking_ang",
        "penalty/torque", "penalty/dof_acc", "penalty/collision",
        "penalty/action_rate", "penalty/feet_slip", "penalty/impact_reduction",
        "penalty/stumble", "penalty/ang_vel_xy",
    )
    REWARD_GROUP_KEYS = ("reward_group/style", "reward_group/task")

    def __init__(self, num_envs, device, xml_path="scene_flat.xml", max_ep_len=500,
                 resample_cmd=True, preset=None, use_graph=None, seed=0, render_size=None):
        self.N = num_envs
        self.dev = device
        self.wp_dev = wp.get_device(f"cuda:{device.index}")
        self.max_steps = max_ep_len
        self.resample_cmd = resample_cmd
        self.frame_skip = 10
        self.rng = torch.Generator(device=device).manual_seed(seed)

        mjm = mujoco.MjModel.from_xml_path(xml_path)
        if mjm.nu != 12:
            raise RuntimeError(f"Go2の12 actuatorを想定していますが、MJCFは nu={mjm.nu} です")

        # APEXはURDF由来のtorque limitで最終torqueをclipする。
        # MJCF側に有効なforce/ctrl rangeがあればそれを優先し、無ければGo2 URDF値へfallbackする。
        fallback_tau = np.asarray([23.7, 23.7, 45.43] * 4, dtype=np.float64)
        force_range = np.asarray(mjm.actuator_forcerange, dtype=np.float64)
        force_limited = np.asarray(mjm.actuator_forcelimited).reshape(-1) > 0
        force_lim = np.max(np.abs(force_range), axis=1)
        ctrl_range = np.asarray(mjm.actuator_ctrlrange, dtype=np.float64)
        ctrl_limited = np.asarray(mjm.actuator_ctrllimited).reshape(-1) > 0
        ctrl_lim = np.max(np.abs(ctrl_range), axis=1)
        valid_force = force_limited & np.isfinite(force_lim) & (force_lim > 1.0) & (force_lim < 1e4)
        valid_ctrl = ctrl_limited & np.isfinite(ctrl_lim) & (ctrl_lim > 1.0) & (ctrl_lim < 1e4)
        torque_limits_np = np.where(valid_force, force_lim, np.where(valid_ctrl, ctrl_lim, fallback_tau))

        # ctrlを直接torqueとして使うこのMJCF実装では、同じlimitをMuJoCo側にも設定する。
        mjm.actuator_ctrllimited[:] = 1
        mjm.actuator_ctrlrange[:, 0] = -torque_limits_np
        mjm.actuator_ctrlrange[:, 1] = torque_limits_np
        for k, v in PHYSICS_PRESETS[PHYSICS_PRESET if preset is None else preset].items():
            setattr(mjm.opt, k, v)
        mjm.opt.disableflags |= (mujoco.mjtDisableBit.mjDSBL_MULTICCD
                                 | mujoco.mjtDisableBit.mjDSBL_NATIVECCD)
        self.render_w, self.render_h = render_size or VIDEO_PANEL
        mjm.vis.global_.offwidth = max(mjm.vis.global_.offwidth, self.render_w)
        mjm.vis.global_.offheight = max(mjm.vis.global_.offheight, self.render_h)
        self.mjm = mjm
        self.mjd = mujoco.MjData(mjm)
        self.dt = self.frame_skip * mjm.opt.timestep
        self.renderer = None

        mujoco.mj_resetData(mjm, self.mjd)
        mujoco.mj_forward(mjm, self.mjd)
        with wp.ScopedDevice(self.wp_dev):
            self.m = mjw.put_model(mjm)
            self.m.opt.warn_overflow = 0 #エラー文非表示
            self.m.opt.graph_conditional = False        # 条件グラフノードは CUDA 12.4 未満で使えない
            # collision / feet slip / stumble用のcfrc_extを更新する。
            # APEX(Isaac Gym)のnet_contact_force_tensorに相当する情報をMuJoCo-Warp側で得るため。
            self.m.opt.run_rne_postconstraint = True
            self.d = mjw.put_data(mjm, self.mjd, nworld=self.N)
            self.perm_wp = wp.array(self.PERM.astype(np.int32), dtype=wp.int32)
            self.target_wp = wp.zeros((self.N, 12), dtype=float)
            self.ref_target_wp = wp.zeros((self.N, 12), dtype=float)
            self.prior_factor_wp = wp.zeros(self.N, dtype=float)
            self.torque_limit_wp = wp.array(torque_limits_np.astype(np.float32), dtype=float)

        self.qpos = wp.to_torch(self.d.qpos)
        self.qvel = wp.to_torch(self.d.qvel)
        self.sensordata = wp.to_torch(self.d.sensordata)
        self.ctrl = wp.to_torch(self.d.ctrl)
        self.cfrc_ext = wp.to_torch(self.d.cfrc_ext)
        self.target = wp.to_torch(self.target_wp)
        self.ref_target = wp.to_torch(self.ref_target_wp)
        self.prior_factor = wp.to_torch(self.prior_factor_wp)
        self.torque_limits = torch.as_tensor(torque_limits_np, dtype=torch.float32, device=device)
        print("torque limits [Nm]:", np.round(torque_limits_np, 2).tolist())

        self.joints = ['FR_hip', 'FR_thigh', 'FR_calf', 'FL_hip', 'FL_thigh', 'FL_calf',
                       'RR_hip', 'RR_thigh', 'RR_calf', 'RL_hip', 'RL_thigh', 'RL_calf']
        adr = lambda n: mjm.sensor_adr[mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_SENSOR, n)]
        self.a_pos = adr(f"{self.joints[0]}_pos")
        self.a_vel = adr(f"{self.joints[0]}_vel")
        self.a_gyro = adr("imu_gyro")

        t32 = lambda x: torch.as_tensor(x, dtype=torch.float32, device=device)
        self.perm = torch.as_tensor(self.PERM, dtype=torch.long, device=device)
        self.cmd_scale = t32(CMD_SCALE)
        self.actor_cmd_scale = t32([2.0, 2.0, 0.25])
        self.act_scale = t32(ACT_SCALE)
        self.standing = t32(standing_pose(H0))
        self.default_sens = self.standing[self.perm]
        self.jnt_lo = t32(mjm.jnt_range[1:, 0])[self.perm]
        self.jnt_hi = t32(mjm.jnt_range[1:, 1])[self.perm]
        self.gravity_world = t32([0.0, 0.0, -1.0])

        # Go2脚FK定数（LEGS順 = FL,FR,RL,RR = qpos順）
        self.leg_hx = _HX.to(device=device, dtype=torch.float32)
        self.leg_hy = _HY.to(device=device, dtype=torch.float32)
        self.leg_s = _S.to(device=device, dtype=torch.float32)

        self.trot = BatchTrot(device)
        self.cmd = torch.zeros(self.N, 3, device=device)
        self.h_swing = torch.zeros(self.N, device=device)
        self.q_tab = torch.zeros(self.N, CYC, 12, device=device)
        self.dq_tab = torch.zeros(self.N, CYC, 12, device=device)
        self.k0 = torch.zeros(self.N, dtype=torch.long, device=device)
        self.step_count = torch.zeros(self.N, dtype=torch.long, device=device)
        self.last_target = torch.zeros(self.N, 12, device=device)
        self.actions = torch.zeros(self.N, 12, device=device)
        self.last_actions = torch.zeros(self.N, 12, device=device)
        self.last_dof_vel = torch.zeros(self.N, 12, device=device)
        self.last_contacts = torch.zeros(self.N, 4, dtype=torch.bool, device=device)
        self.last_foot_world = torch.zeros(self.N, 4, 3, device=device)
        self.last_foot_vel = torch.zeros(self.N, 4, 3, device=device)
        self.all_idx = torch.arange(self.N, device=device)

        self._setup_contact_body_ids()

        # APEX標準のreference-free Actorは45次元。足先位置/reference/phaseはActorへ入れない。
        self.obs_dim = 45
        # APEX公式77D privileged critic:
        # base lin vel(3) + Actorに含まれる残り42D + phase(1) + ref joint(12) + ref foot(12) + ref quat(4)
        self.critic_obs_dim = 77
        self.act_dim, self.act_limit = 12, float(ACTION_CLIP)
        self.cmd_resample_steps = max(1, int(round(CMD_RESAMPLE_TIME / self.dt)))
        self.graph = self.fwd_graph = None
        use_graph = USE_CUDA_GRAPH if use_graph is None else use_graph
        if use_graph:
            self._build_graph()

    def _setup_contact_body_ids(self):
        names = []
        for i in range(self.mjm.nbody):
            n = mujoco.mj_id2name(self.mjm, mujoco.mjtObj.mjOBJ_BODY, i)
            names.append(n or "")

        foot_ids = []
        separate_foot = True
        for leg in ("FL", "FR", "RL", "RR"):
            exact = [i for i, n in enumerate(names) if n == f"{leg}_foot"]
            if exact:
                foot_ids.append(exact[0])
                continue
            # MJCFによってはfoot geomがcalf bodyに直接付いている。その場合はcalf forceを足接触として使う。
            calf = [i for i, n in enumerate(names) if n == f"{leg}_calf"]
            if calf:
                foot_ids.append(calf[0])
                separate_foot = False
            else:
                foot_ids = []
                break

        self.foot_contact_body_ids = (
            torch.as_tensor(foot_ids, dtype=torch.long, device=self.dev) if len(foot_ids) == 4 else None
        )
        self.separate_foot_bodies = bool(separate_foot and len(foot_ids) == 4)

        penalized = []
        for i, name in enumerate(names):
            low = name.lower()
            hit = ("base" in low) or ("trunk" in low) or low.endswith("_hip") or low.endswith("_thigh")
            # footが独立bodyならcalf衝突もAPEXと同様に罰する。
            # foot geomがcalfに付くMJCFでは、calfを罰すると通常の接地までcollisionになるため除外する。
            if self.separate_foot_bodies:
                hit = hit or low.endswith("_calf")
            if hit and i != 0:
                penalized.append(i)
        self.penalized_body_ids = (
            torch.as_tensor(sorted(set(penalized)), dtype=torch.long, device=self.dev)
            if penalized else None
        )

        termination = []
        for i, name in enumerate(names):
            low = name.lower()
            hit = ("base" in low) or ("trunk" in low) or low.endswith("_hip")
            if hit and i != 0:
                termination.append(i)
        self.termination_body_ids = (
            torch.as_tensor(sorted(set(termination)), dtype=torch.long, device=self.dev)
            if termination else None
        )

        if self.foot_contact_body_ids is None:
            print("[APEX reward warning] foot/calf bodyを特定できないため feet_slip / stumble は0として扱います")
        else:
            print("APEX contact bodies:", [names[i] for i in foot_ids])
        if self.penalized_body_ids is None:
            print("[APEX reward warning] collision対象bodyを特定できないため collision penalty は0として扱います")

    def _stream(self):
        return wp.stream_from_torch(torch.cuda.current_stream(self.dev))

    def _launch_pd(self):
        wp.launch(_pd_kernel, dim=(self.N, 12),
                  inputs=[self.d.qpos, self.d.qvel, self.target_wp, self.ref_target_wp,
                          self.prior_factor_wp, self.perm_wp, self.torque_limit_wp,
                          float(SERVO_KP), float(SERVO_KD)],
                  outputs=[self.d.ctrl], device=self.wp_dev)

    def _build_graph(self):
        with wp.ScopedDevice(self.wp_dev):
            for _ in range(2):
                self._launch_pd(); mjw.step(self.m, self.d)
            mjw.forward(self.m, self.d); wp.synchronize()
            with wp.ScopedCapture() as cap:
                for _ in range(self.frame_skip):
                    self._launch_pd(); mjw.step(self.m, self.d)
            self.graph = cap.graph
            with wp.ScopedCapture() as cap_fwd:
                mjw.forward(self.m, self.d)
            self.fwd_graph = cap_fwd.graph

    def _physics(self):
        if self.graph is not None:
            wp.capture_launch(self.graph, stream=self._stream())
        else:
            with wp.ScopedDevice(self.wp_dev), wp.ScopedStream(self._stream()):
                for _ in range(self.frame_skip):
                    self._launch_pd(); mjw.step(self.m, self.d)

    def _forward(self):
        if self.fwd_graph is not None:
            wp.capture_launch(self.fwd_graph, stream=self._stream())
        else:
            with wp.ScopedDevice(self.wp_dev), wp.ScopedStream(self._stream()):
                mjw.forward(self.m, self.d)

    def _sample_cmd(self, n):
        c = (torch.rand(n, 3, device=self.dev, generator=self.rng) * 2 - 1) * self.cmd_scale
        still = torch.rand(n, 1, device=self.dev, generator=self.rng) < 0.15
        return torch.where(still, torch.zeros_like(c), c)

    def _set_cmd(self, idx, cmd):
        cmd = torch.as_tensor(cmd, dtype=torch.float32, device=self.dev).reshape(-1, 3).clone()
        if cmd.shape[0] == 1:
            cmd = cmd.expand(idx.numel(), 3).clone()
        dead = (cmd / self.cmd_scale).norm(dim=1) < 0.05
        cmd[dead] = 0.0
        moving = cmd.abs().any(dim=1)
        hs = torch.where(moving, torch.full_like(cmd[:, 0], H_SWING), torch.zeros_like(cmd[:, 0]))
        self.cmd[idx], self.h_swing[idx] = cmd, hs
        q, dq = self.trot.joint_cycle(cmd.double(), hs.double(), fps=int(round(1 / self.dt)))
        self.q_tab[idx], self.dq_tab[idx] = q.float(), dq.float()

    @staticmethod
    def _quat2mat(q):
        w, x, y, z = q.unbind(-1)
        return torch.stack([
            1 - 2*(y*y + z*z), 2*(x*y - z*w), 2*(x*z + y*w),
            2*(x*y + z*w), 1 - 2*(x*x + z*z), 2*(y*z - x*w),
            2*(x*z - y*w), 2*(y*z + x*w), 1 - 2*(x*x + y*y),
        ], dim=-1).reshape(-1, 3, 3)

    def _projected_gravity(self, Rm=None):
        if Rm is None:
            Rm = self._quat2mat(self.qpos[:, 3:7])
        g = self.gravity_world.unsqueeze(0).expand(self.N, 3)
        return torch.einsum('nij,nj->ni', Rm.transpose(1, 2), g)

    def _foot_pos_body_fk(self, q):
        """qpos順(FL,FR,RL,RR)の12関節角から4脚足先位置をbase frameで計算。"""
        qq = q.reshape(-1, 4, 3)
        t1, t2, t3 = qq[..., 0], qq[..., 1], qq[..., 2]
        a = t2 + t3
        x = -float(L1) * torch.sin(t2) - 0.002 * torch.cos(a) - 0.213 * torch.sin(a)
        y0 = self.leg_s.unsqueeze(0) * float(D_HIP)
        z0 = -float(L1) * torch.cos(t2) + 0.002 * torch.sin(a) - 0.213 * torch.cos(a)
        ct, st = torch.cos(t1), torch.sin(t1)
        y = ct * y0 - st * z0
        z = st * y0 + ct * z0
        x = x + self.leg_hx.unsqueeze(0)
        y = y + self.leg_hy.unsqueeze(0)
        return torch.stack([x, y, z], dim=-1)

    def _foot_world_from_state(self, idx=None):
        if idx is None:
            qbase = self.qpos[:, 3:7]
            base = self.qpos[:, :3]
            qj = self.qpos[:, 7:]
        else:
            qbase = self.qpos[idx, 3:7]
            base = self.qpos[idx, :3]
            qj = self.qpos[idx, 7:]
        Rm = self._quat2mat(qbase)
        pbody = self._foot_pos_body_fk(qj)
        return base.unsqueeze(1) + torch.einsum('nij,nkj->nki', Rm, pbody)

    def _current_foot_heading(self, Rm=None):
        """APEXのquat_apply_yaw(quat_conjugate(base_quat), foot-base)相当。"""
        if Rm is None:
            Rm = self._quat2mat(self.qpos[:, 3:7])
        pbody = self._foot_pos_body_fk(self.qpos[:, 7:])
        rel_world = torch.einsum('nij,nkj->nki', Rm, pbody)
        yaw = torch.atan2(Rm[:, 1, 0], Rm[:, 0, 0])
        c, s = torch.cos(yaw).unsqueeze(1), torch.sin(yaw).unsqueeze(1)
        xh = c * rel_world[..., 0] + s * rel_world[..., 1]
        yh = -s * rel_world[..., 0] + c * rel_world[..., 1]
        return torch.stack([xh, yh, rel_world[..., 2]], dim=-1)

    def _ground_height(self, x, y):
        return torch.zeros_like(x)

    def _reset_idx(self, idx, command=None):
        n = idx.numel()
        if n == 0:
            return
        self._set_cmd(idx, self._sample_cmd(n) if command is None else command)
        self.step_count[idx] = 0

        if USE_RSI:
            self.k0[idx] = torch.randint(CYC, (n,), device=self.dev, generator=self.rng)
            t0 = (self.k0[idx] * self.dt).double()
            cmd_d, hs_d = self.cmd[idx].double(), self.h_swing[idx].double()
            qp = self.trot.qpos_at(cmd_d, t0, hs_d).float()
            qv = self.trot.qvel_at(cmd_d, t0, hs_d).float()
            qp[:, 7:] += torch.rand(n, 12, device=self.dev, generator=self.rng) * 0.1 - 0.05
        else:
            self.k0[idx] = 0
            qp = torch.zeros(n, 19, dtype=torch.float32, device=self.dev)
            qv = torch.zeros(n, 18, dtype=torch.float32, device=self.dev)
            qp[:, 2] = H0
            qp[:, 3] = 1.0
            qp[:, 7:] = self.standing.unsqueeze(0)
            qp[:, 7:] += torch.rand(n, 12, device=self.dev, generator=self.rng) * 0.06 - 0.03

        with wp.ScopedDevice(self.wp_dev), wp.ScopedStream(self._stream()):
            wp.launch(_write_state_kernel, dim=n,
                      inputs=[self.d.qpos, self.d.qvel, self.d.qacc_warmstart,
                              wp.from_torch(qp.contiguous()), wp.from_torch(qv.contiguous()),
                              wp.from_torch(idx.to(torch.int32).contiguous(), dtype=wp.int32), 19, 18],
                      device=self.wp_dev)
        self.last_target[idx] = qp[:, 7:][:, self.perm]
        self.actions[idx] = 0.0
        self.last_actions[idx] = 0.0
        self.last_dof_vel[idx] = qv[:, 6:]
        self.last_contacts[idx] = False
        self.prior_factor[idx] = 0.0

    def _sync_history(self, idx):
        if idx.numel() == 0:
            return
        self.last_actions[idx] = self.actions[idx]
        self.last_dof_vel[idx] = self.qvel[idx, 6:]
        self.last_contacts[idx] = False
        self.last_foot_world[idx] = self._foot_world_from_state(idx)
        self.last_foot_vel[idx] = 0.0

    def reset(self, command=None):
        self._reset_idx(self.all_idx, command)
        self._forward()
        self._sync_history(self.all_idx)
        return self._get_obs()

    def autoreset(self, done, command=None):
        idx = self.all_idx[done]
        if idx.numel():
            self._reset_idx(idx, command)
            self._forward()
            self._sync_history(idx)
        return self._get_obs()

    def _phase_k(self):
        return (self.k0 + self.step_count) % CYC

    def step(self, action, prior_factor=0.0):
        """1 control step. reward_groups shape = [N,2].

        APEXと同様、同じreference index kを制御とrewardの両方に使い、
        reward計算が終わってからphase(step_count)を進める。
        """
        k = self._phase_k()
        q_ref_sens = self.q_tab[self.all_idx, k][:, self.perm]
        self.ref_target.copy_(q_ref_sens)
        pf = float(max(APEX_PRIOR_MIN, min(1.0, prior_factor)))
        self.prior_factor.fill_(pf)

        # APEXはclip_actions=100。関節targetはjoint range、最終torqueはURDF/MJCF limitで制限する。
        self.actions.copy_(torch.clamp(action.to(self.dev, torch.float32), -self.act_limit, self.act_limit))
        target = self.default_sens + self.actions * self.act_scale
        self.last_target = torch.clamp(target, self.jnt_lo, self.jnt_hi)
        self.target.copy_(self.last_target)
        self._physics()

        fallen = self._fallen()
        reward_groups, total_reward, parts = self._compute_reward(fallen, k)

        # reward後に次phaseへ進む。これでAction PriorとStyle rewardのreferenceが1stepずれない。
        self.step_count += 1
        timeout = self.step_count >= self.max_steps
        done = fallen | timeout

        # APEX command resampling_time=5s相当。terminal worldはautoreset側で再サンプルする。
        if self.resample_cmd:
            sel = ((self.step_count % self.cmd_resample_steps) == 0) & (~done)
            idx = self.all_idx[sel]
            if idx.numel():
                self._set_cmd(idx, self._sample_cmd(idx.numel()))

        obs = self._get_obs()

        # 次step用history。reward計算後に更新する。
        self.last_actions.copy_(self.actions)
        self.last_dof_vel.copy_(self.qvel[:, 6:])
        self.last_foot_world.copy_(self._foot_world_from_state())

        return obs, reward_groups, done, {
            "reward_parts": parts,
            "reward_groups": reward_groups,
            "total_reward": total_reward,
            "fallen": fallen,
            "timeout": timeout,
            "cmd": self.cmd,
            "decap_factor": torch.full((self.N,), pf, device=self.dev),
        }

    def _get_obs(self):
        """APEX標準45D Actor obs。足先位置/reference/phaseは含めない。"""
        sd = self.sensordata
        q_err = sd[:, self.a_pos:self.a_pos + 12] - self.default_sens
        dq = sd[:, self.a_vel:self.a_vel + 12]
        base_ang_vel = sd[:, self.a_gyro:self.a_gyro + 3] * 0.25
        projected_gravity = self._projected_gravity()
        cmd_obs = self.cmd * self.actor_cmd_scale
        return torch.cat([
            base_ang_vel,
            projected_gravity,
            cmd_obs,
            q_err,
            dq * 0.05,
            self.actions,
        ], dim=1)

    def get_critic_obs(self):
        """APEX公式77D相当のasymmetric privileged critic observation。"""
        obs = self._get_obs()
        k = self._phase_k()
        q_ref_qpos = self.q_tab[self.all_idx, k]
        q_ref_sens = q_ref_qpos[:, self.perm]  # absolute reference joint angles
        ref_foot = self._foot_pos_body_fk(q_ref_qpos).reshape(self.N, 12)
        phase = (k.float() / float(CYC)).unsqueeze(1)
        ref_quat = torch.zeros(self.N, 4, dtype=torch.float32, device=self.dev)
        ref_quat[:, 0] = 1.0  # procedural trotはbase姿勢referenceを持たないためupright identityを使用
        Rm = self._quat2mat(self.qpos[:, 3:7])
        base_lin_vel = torch.einsum('nij,nj->ni', Rm.transpose(1, 2), self.qvel[:, :3]) * 2.0
        # 45 Actor obs + 3 base lin vel + 1 phase + 12 ref joint + 12 ref foot + 4 ref quat = 77
        return torch.cat([base_lin_vel, obs, phase, q_ref_sens, ref_foot, ref_quat], dim=1)

    def _contact_reward_terms(self, current_foot_world):
        zero = torch.zeros(self.N, device=self.dev)
        foot_vel = (current_foot_world - self.last_foot_world) / self.dt

        if self.foot_contact_body_ids is None:
            feet_slip = zero
            stumble = zero
        else:
            foot_force = self.cfrc_ext[:, self.foot_contact_body_ids, 3:6]
            contact = foot_force[..., 2] > 1.0
            contact_filt = torch.logical_or(contact, self.last_contacts)
            self.last_contacts.copy_(contact)
            feet_slip = torch.sum(contact_filt.float() * torch.sum(foot_vel[..., :2] ** 2, dim=-1), dim=1)
            stumble = torch.any(
                torch.norm(foot_force[..., :2], dim=-1) > 5.0 * torch.abs(foot_force[..., 2]), dim=1
            ).float()

        if self.penalized_body_ids is None:
            collision = zero
        else:
            pf = self.cfrc_ext[:, self.penalized_body_ids, 3:6]
            collision = torch.sum((torch.norm(pf, dim=-1) > 0.1).float(), dim=1)
        return feet_slip, stumble, collision, foot_vel

    def _compute_reward(self, fallen, k):
        del fallen  # APEX current configのtermination scaleは0。done自体は別途使う。
        q = self.qpos[:, 7:]
        dq = self.qvel[:, 6:]
        q_ref = self.q_tab[self.all_idx, k]

        # APEX: linear commandがほぼ0ならreferenceではなくdefault standing poseを模倣targetにする。
        is_standing = torch.norm(self.cmd[:, :2], dim=1) < 0.1
        q_target = torch.where(is_standing.unsqueeze(1), self.standing.unsqueeze(0), q_ref)

        # ---- Style / imitation critic ----
        e_joint = torch.mean((q - q_target) ** 2, dim=1)
        r_joint_raw = torch.exp(-e_joint / APEX_SIGMA_IMIT_ANGLES)

        Rm = self._quat2mat(self.qpos[:, 3:7])
        current_foot_heading = self._current_foot_heading(Rm)
        ref_foot = self._foot_pos_body_fk(q_target)
        e_foot = torch.sum((ref_foot - current_foot_heading) ** 2, dim=(1, 2))
        r_foot_raw = torch.exp(-e_foot / APEX_SIGMA_IMIT_FOOT)

        # procedural referenceにはbase quaternionデータが無いのでupright identityをreferenceにする。
        ref_quat = torch.zeros_like(self.qpos[:, 3:7])
        ref_quat[:, 0] = 1.0
        e_quat = torch.sum((ref_quat - self.qpos[:, 3:7]) ** 2, dim=1)
        r_quat_raw = torch.exp(-e_quat / APEX_SIGMA_IMIT_QUAT)

        # ---- Task critic ----
        v_body = torch.einsum('nij,nj->ni', Rm.transpose(1, 2), self.qvel[:, 0:3])
        base_ang_vel = self.sensordata[:, self.a_gyro:self.a_gyro + 3]
        lin_err = torch.sum((self.cmd[:, :2] - v_body[:, :2]) ** 2, dim=1)
        yaw_err = (self.cmd[:, 2] - base_ang_vel[:, 2]) ** 2
        r_lin_raw = torch.exp(-lin_err / APEX_TRACKING_SIGMA)
        r_ang_raw = torch.exp(-yaw_err / APEX_TRACKING_SIGMA)

        torque_raw = torch.sum(self.ctrl ** 2, dim=1)
        dof_acc_raw = torch.sum(((self.last_dof_vel - dq) / self.dt) ** 2, dim=1)
        action_rate_raw = torch.sum((self.last_actions - self.actions) ** 2, dim=1)
        ang_vel_xy_raw = torch.sum(base_ang_vel[:, :2] ** 2, dim=1)

        current_foot_world = self._foot_world_from_state()
        feet_slip_raw, stumble_raw, collision_raw, foot_vel = self._contact_reward_terms(current_foot_world)
        delta_vz2 = (foot_vel[..., 2] - self.last_foot_vel[..., 2]) ** 2
        impact_raw = torch.sum(torch.clamp(delta_vz2, max=2.0), dim=1)
        self.last_foot_vel.copy_(foot_vel)

        # APEX _prepare_reward_function() と同様、scaleにcontrol dtを掛けた1step寄与へ変換。
        r_joint = self.dt * W_IMIT_ANGLES * r_joint_raw
        r_foot = self.dt * W_IMIT_FOOT * r_foot_raw
        r_quat = self.dt * W_IMIT_QUAT * r_quat_raw
        r_lin = self.dt * W_TRACK_LIN * r_lin_raw
        r_ang = self.dt * W_TRACK_ANG * r_ang_raw
        p_torque = self.dt * W_TORQUE * torque_raw
        p_dof_acc = self.dt * W_DOF_ACC * dof_acc_raw
        p_collision = self.dt * W_COLLISION * collision_raw
        p_action_rate = self.dt * W_ACTION_RATE * action_rate_raw
        p_feet_slip = self.dt * W_FEET_SLIP * feet_slip_raw
        p_impact = self.dt * W_IMPACT_REDUCTION * impact_raw

        rough = bool(getattr(self, "terrain_curriculum", False))
        p_stumble = self.dt * (W_STUMBLE_ROUGH if rough else 0.0) * stumble_raw
        p_ang_vel_xy = self.dt * (W_ANG_VEL_XY_ROUGH if rough else W_ANG_VEL_XY_FLAT) * ang_vel_xy_raw

        style = r_joint + r_foot + r_quat
        task = (r_lin + r_ang
                - p_torque - p_dof_acc - p_collision - p_action_rate
                - p_feet_slip - p_impact - p_stumble - p_ang_vel_xy)
        reward_groups = torch.stack([style, task], dim=1)
        total = reward_groups.sum(1)
        parts = torch.stack([
            r_joint, r_foot, r_quat, r_lin, r_ang,
            p_torque, p_dof_acc, p_collision, p_action_rate,
            p_feet_slip, p_impact, p_stumble, p_ang_vel_xy,
        ], dim=1)
        return reward_groups, total, parts

    def _termination_contact_fallen(self):
        if self.termination_body_ids is None:
            return torch.zeros(self.N, dtype=torch.bool, device=self.dev)
        f = self.cfrc_ext[:, self.termination_body_ids, 3:6]
        return torch.any(torch.norm(f, dim=-1) > 1.0, dim=1)

    def _fallen(self):
        geometric = ((self.qpos[:, 2] < 0.18) | (self.qpos[:, 2] > 0.55) |
                     (self.qpos[:, 3].abs() < 0.55))
        return geometric | self._termination_contact_fallen()

    def close(self):
        if self.renderer is not None:
            self.renderer.close(); self.renderer = None

    def render(self, world=0):
        if self.renderer is None:
            self.renderer = mujoco.Renderer(self.mjm, height=self.render_h, width=self.render_w)
        q = self.qpos[world].detach().cpu().numpy()
        v = self.qvel[world].detach().cpu().numpy()
        self.mjd.qpos[:] = q
        self.mjd.qvel[:] = v
        mujoco.mj_forward(self.mjm, self.mjd)
        self.renderer.update_scene(self.mjd)
        return self.renderer.render()


## APEX Terrain Curriculum付きMuJoCo-Warp環境

`Go2ImitationWarpEnv` を継承し、APEX公式のrough terrain運用を追加します。

- 並列envごとにterrain column（種類）を固定
- terrain level（難易度）だけをepisode reset時に上下
- 4 m以上進めたら level +1
- 指令速度から期待される距離の半分未満なら level -1
- Actorにはheight mapを渡さない
- Criticには17×11=187点のterrain heightをprivileged observationとして追加可能


In [8]:

class Go2ApexTerrainWarpEnv(Go2ImitationWarpEnv):
    TERRAIN_TYPES = ("smooth_slope", "rough_slope", "stairs_down", "stairs_up", "discrete")

    def __init__(self, *args, terrain_meta=None, terrain_curriculum=False, **kwargs):
        self.terrain_meta = terrain_meta
        self.terrain_curriculum = bool(terrain_curriculum and terrain_meta is not None)
        super().__init__(*args, **kwargs)

        if self.terrain_curriculum:
            self.terrain_height = torch.as_tensor(terrain_meta["height_m"], dtype=torch.float32, device=self.dev)
            self.terrain_origins_all = torch.as_tensor(terrain_meta["origins"], dtype=torch.float32, device=self.dev)
            self.max_terrain_level = int(terrain_meta["rows"])
            self.max_init_terrain_level = int(terrain_meta["max_init_level"])
            self.terrain_cols_count = int(terrain_meta["cols"])
            self.terrain_hscale = float(terrain_meta["horizontal_scale"])
            self.terrain_border = float(terrain_meta["border_size"])
            self.terrain_env_length = float(terrain_meta["terrain_length"])

            # APEX: terrain_types = floor(arange(N) / (N/num_cols))
            self.terrain_types = torch.floor(
                torch.arange(self.N, device=self.dev, dtype=torch.float32)
                * self.terrain_cols_count / self.N
            ).long().clamp(max=self.terrain_cols_count - 1)
            self.terrain_levels = torch.randint(
                low=0, high=min(self.max_init_terrain_level + 1, self.max_terrain_level),
                size=(self.N,), device=self.dev, generator=self.rng
            )
            self.env_origins = self.terrain_origins_all[self.terrain_levels, self.terrain_types].clone()
            self._terrain_init_done = False

            # APEX Go2 configと同じ17×11点。ActorではなくCritic用。
            xs = torch.arange(-0.8, 0.8001, 0.1, device=self.dev)
            ys = torch.arange(-0.5, 0.5001, 0.1, device=self.dev)
            gx, gy = torch.meshgrid(xs, ys, indexing="ij")
            self.height_points_xy = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1)
            self.num_height_points = self.height_points_xy.shape[0]
            if TERRAIN_MEASURE_HEIGHTS_FOR_CRITIC:
                self.critic_obs_dim += self.num_height_points
        else:
            self.terrain_levels = torch.zeros(self.N, dtype=torch.long, device=self.dev)
            self.terrain_types = torch.zeros(self.N, dtype=torch.long, device=self.dev)
            self.env_origins = torch.zeros(self.N, 3, device=self.dev)
            self._terrain_init_done = False
            self.num_height_points = 0

    def _ground_height(self, x, y):
        if not self.terrain_curriculum:
            return torch.zeros_like(x)
        ix = torch.floor((x + self.terrain_border) / self.terrain_hscale).long().clamp(0, self.terrain_height.shape[0]-2)
        iy = torch.floor((y + self.terrain_border) / self.terrain_hscale).long().clamp(0, self.terrain_height.shape[1]-2)
        # APEX _get_heights と同様に近傍3点のminを用いる
        h1 = self.terrain_height[ix, iy]
        h2 = self.terrain_height[ix+1, iy]
        h3 = self.terrain_height[ix, iy+1]
        return torch.minimum(torch.minimum(h1, h2), h3)

    def _measure_terrain_heights(self):
        if not self.terrain_curriculum:
            return torch.zeros(self.N, 0, device=self.dev)
        q = self.qpos[:, 3:7]  # MuJoCo freejoint quat: w,x,y,z
        w, x, y, z = q.unbind(1)
        yaw = torch.atan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))
        c, s = torch.cos(yaw), torch.sin(yaw)
        px = self.height_points_xy[:, 0].unsqueeze(0)
        py = self.height_points_xy[:, 1].unsqueeze(0)
        wx = self.qpos[:, 0:1] + c[:,None]*px - s[:,None]*py
        wy = self.qpos[:, 1:2] + s[:,None]*px + c[:,None]*py
        return self._ground_height(wx, wy)

    def _update_terrain_curriculum(self, idx):
        if not self.terrain_curriculum or not self._terrain_init_done or idx.numel() == 0:
            return
        distance = torch.norm(self.qpos[idx, :2] - self.env_origins[idx, :2], dim=1)
        move_up = distance > (self.terrain_env_length / 2.0)
        max_episode_s = self.max_steps * self.dt
        required = torch.norm(self.cmd[idx, :2], dim=1) * max_episode_s * 0.5
        move_down = (distance < required) & (~move_up)
        self.terrain_levels[idx] += move_up.long() - move_down.long()
        # APEX: last levelを解いたrobotはrandom valid levelへ戻す
        solved = self.terrain_levels[idx] >= self.max_terrain_level
        clipped = self.terrain_levels[idx].clamp(min=0)
        if bool(solved.any()):
            clipped[solved] = torch.randint(
                self.max_terrain_level, (int(solved.sum().item()),),
                device=self.dev, generator=self.rng
            )
        self.terrain_levels[idx] = clipped
        self.env_origins[idx] = self.terrain_origins_all[self.terrain_levels[idx], self.terrain_types[idx]]

    def _reset_idx(self, idx, command=None):
        if not self.terrain_curriculum:
            return super()._reset_idx(idx, command)
        n = idx.numel()
        if n == 0:
            return
        self._set_cmd(idx, self._sample_cmd(n) if command is None else command)
        self.step_count[idx] = 0
        origin = self.env_origins[idx]
        jitter = torch.rand(n, 2, device=self.dev, generator=self.rng) * 2.0 - 1.0

        if USE_RSI:
            self.k0[idx] = torch.randint(CYC, (n,), device=self.dev, generator=self.rng)
            t0 = (self.k0[idx] * self.dt).double()
            cmd_d, hs_d = self.cmd[idx].double(), self.h_swing[idx].double()
            qp = self.trot.qpos_at(cmd_d, t0, hs_d).float()
            qv = self.trot.qvel_at(cmd_d, t0, hs_d).float()
            qp[:, 0:2] += origin[:, 0:2] + jitter
            qp[:, 2] += origin[:, 2]
            qp[:, 7:] += torch.rand(n, 12, device=self.dev, generator=self.rng) * 0.1 - 0.05
        else:
            self.k0[idx] = 0
            qp = torch.zeros(n, 19, dtype=torch.float32, device=self.dev)
            qv = torch.zeros(n, 18, dtype=torch.float32, device=self.dev)
            qp[:, 0:2] = origin[:, 0:2] + jitter
            qp[:, 2] = H0 + origin[:, 2]
            qp[:, 3] = 1.0
            qp[:, 7:] = self.standing.unsqueeze(0)
            qp[:, 7:] += torch.rand(n, 12, device=self.dev, generator=self.rng) * 0.06 - 0.03

        with wp.ScopedDevice(self.wp_dev), wp.ScopedStream(self._stream()):
            wp.launch(_write_state_kernel, dim=n,
                      inputs=[self.d.qpos, self.d.qvel, self.d.qacc_warmstart,
                              wp.from_torch(qp.contiguous()), wp.from_torch(qv.contiguous()),
                              wp.from_torch(idx.to(torch.int32).contiguous(), dtype=wp.int32), 19, 18],
                      device=self.wp_dev)
        self.last_target[idx] = qp[:, 7:][:, self.perm]
        self.actions[idx] = 0.0
        self.last_actions[idx] = 0.0
        self.last_dof_vel[idx] = qv[:, 6:]
        self.last_contacts[idx] = False
        self.prior_factor[idx] = 0.0

    def reset(self, command=None):
        self._reset_idx(self.all_idx, command)
        self._forward()
        self._sync_history(self.all_idx)
        self._terrain_init_done = True
        return self._get_obs()

    def autoreset(self, done, command=None):
        idx = self.all_idx[done]
        if idx.numel():
            self._update_terrain_curriculum(idx)
            self._reset_idx(idx, command)
            self._forward()
            self._sync_history(idx)
        return self._get_obs()

    def get_critic_obs(self):
        base = super().get_critic_obs()
        if self.terrain_curriculum and TERRAIN_MEASURE_HEIGHTS_FOR_CRITIC:
            measured = self._measure_terrain_heights()
            # APEXのprivileged height inputと同様、baseから見た相対値をclip
            # APEX obs_scales.height_measurements = 5.0
            hobs = torch.clamp(self.qpos[:, 2:3] - 0.5 - measured, -1.0, 1.0) * 5.0
            return torch.cat([base, hobs], dim=1)
        return base

    def _fallen(self):
        if not self.terrain_curriculum:
            return super()._fallen()
        ground = self._ground_height(self.qpos[:, 0], self.qpos[:, 1])
        rel_z = self.qpos[:, 2] - ground
        geometric = (rel_z < 0.18) | (rel_z > 0.55) | (self.qpos[:, 3].abs() < 0.55)
        return geometric | self._termination_contact_fallen()

    def step(self, action, prior_factor=0.0):
        obs, reward_groups, done, info = super().step(action, prior_factor)
        info["terrain_level"] = self.terrain_levels
        info["terrain_type_col"] = self.terrain_types
        return obs, reward_groups, done, info

    def render(self, world=0):
        if self.renderer is None:
            self.renderer = mujoco.Renderer(self.mjm, height=self.render_h, width=self.render_w)
        q = self.qpos[world].detach().cpu().numpy()
        v = self.qvel[world].detach().cpu().numpy()
        self.mjd.qpos[:] = q
        self.mjd.qvel[:] = v
        mujoco.mj_forward(self.mjm, self.mjd)
        cam = mujoco.MjvCamera()
        mujoco.mjv_defaultCamera(cam)
        cam.type = mujoco.mjtCamera.mjCAMERA_FREE
        cam.lookat[:] = q[:3]
        cam.distance = 2.2
        cam.azimuth = 135
        cam.elevation = -20
        self.renderer.update_scene(self.mjd, camera=cam)
        return self.renderer.render()


## 7. SNN Actor + 2 Critic + PPO Rollout Storage


In [9]:
class RunningMeanStd:
    """PPO用の観測正規化。rolloutに格納するのは正規化済み観測なのでPPO ratioが崩れない。"""
    def __init__(self, dim, device, clip_limit=50.0):
        self.mean = torch.zeros(dim, device=device)
        self.var = torch.ones(dim, device=device)
        self.count = torch.tensor(1e-4, device=device)
        self.clip_limit = clip_limit

    @torch.no_grad()
    def update(self, x):
        if x.numel() == 0:
            return
        x = x.float()
        bmean = x.mean(0)
        bvar = x.var(0, unbiased=False)
        n = torch.tensor(float(x.shape[0]), device=x.device)
        total = self.count + n
        delta = bmean - self.mean
        new_mean = self.mean + delta * n / total
        m_a = self.var * self.count
        m_b = bvar * n
        m2 = m_a + m_b + delta.square() * self.count * n / total
        self.mean, self.var, self.count = new_mean, m2 / total, total

    def normalize(self, x):
        return torch.clamp((x - self.mean) / torch.sqrt(self.var + 1e-6),
                           -self.clip_limit, self.clip_limit)

    def state_dict(self):
        return {"mean": self.mean.detach().cpu(), "var": self.var.detach().cpu(),
                "count": self.count.detach().cpu()}


# ==================== Spike generation and LIF ====================
LIF_BETA, LIF_THRESHOLD, SPIKE_SLOPE = 0.5, 1.0, 3.0
ENC_VTH = 0.999


class PopSpike(torch.autograd.Function):
    @staticmethod
    def forward(ctx, p, u, slope, vth):
        v = p.unsqueeze(-1) + u
        ctx.save_for_backward(v)
        ctx.slope, ctx.vth = slope, vth
        return v.gt(vth).to(p.dtype).reshape(p.shape[0], -1)

    @staticmethod
    def backward(ctx, grad_output):
        (v,) = ctx.saved_tensors
        denom = (ctx.slope * (v - ctx.vth).abs() + 1.0) ** 2
        return (grad_output.view(v.shape) / denom).sum(-1), None, None, None


class SpikeFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, slope):
        ctx.save_for_backward(x); ctx.slope = slope
        return (x > 0).to(x.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output / (ctx.slope * x.abs() + 1.0) ** 2, None


def lif_step(cur, mem_prev, beta=LIF_BETA, threshold=LIF_THRESHOLD, slope=SPIKE_SLOPE):
    reset = (mem_prev > threshold).to(mem_prev.dtype)
    mem = beta * mem_prev + cur - reset * threshold
    return SpikeFn.apply(mem - threshold, slope), mem


class MLP(nn.Module):
    def __init__(self, sizes):
        super().__init__()
        self.Linear1 = nn.Linear(sizes[0], sizes[1])
        self.Linear2 = nn.Linear(sizes[1], sizes[2])
        self.Linear3 = nn.Linear(sizes[2], sizes[3])
        self.activation = nn.ELU()
        nn.init.kaiming_normal_(self.Linear1.weight)
        nn.init.kaiming_normal_(self.Linear2.weight)
        nn.init.kaiming_normal_(self.Linear3.weight)

    def forward(self, x):
        x = self.activation(self.Linear1(x))
        x = self.activation(self.Linear2(x))
        return self.Linear3(x)


class Encoder(nn.Module):
    """Binary population encoder.

    PPOではlog-probability ratioを再計算するため、同じobsなら同じspike入力になるよう
    deterministic fixed thresholdsを使う。Poisson/random encodingに戻す場合は、
    rollout時のencoder noise自体を保存してPPO更新で再利用する必要がある。
    """
    def __init__(self, obs_dim, pop_dim, device):
        super().__init__()
        self.obs_dim, self.pop_dim = obs_dim, pop_dim
        self.activation = nn.Tanh()
        self.register_buffer("zeros", torch.zeros(1, obs_dim * 2, device=device))
        self.weight = nn.Parameter(torch.ones(1, obs_dim, device=device))
        self.bias = nn.Parameter(torch.zeros(1, obs_dim, device=device))
        # (0, 1)を均等分割。pに比例した個数のbinary spikeが出る。
        u = (torch.arange(pop_dim, device=device, dtype=torch.float32) + 0.5) / pop_dim
        self.register_buffer("fixed_u", u.view(1, 1, pop_dim))

    def forward(self, obs):
        obs = self.activation(obs * self.weight + self.bias)
        p = torch.maximum(torch.cat([obs, -obs], dim=1), self.zeros)
        if PPO_DETERMINISTIC_POP_ENCODING:
            u = self.fixed_u.expand(p.shape[0], p.shape[1], self.pop_dim)
        else:
            u = torch.rand(p.shape[0], p.shape[1], self.pop_dim,
                           device=p.device, dtype=p.dtype)
        return PopSpike.apply(p, u, SPIKE_SLOPE, ENC_VTH)


class Decoder(nn.Module):
    def __init__(self, act_dim, pop_dim, device):
        super().__init__()
        self.act_dim, self.pop_dim = act_dim, pop_dim
        self.activation = nn.Tanh()
        self.register_buffer("weight", torch.ones(1, act_dim, device=device))
        self.register_buffer("bias", torch.zeros(1, act_dim, device=device))

    def forward(self, spk):
        s = spk.reshape(-1, self.act_dim * 2, self.pop_dim).mean(-1)
        return self.activation((s[..., :self.act_dim] - s[..., self.act_dim:]) * self.weight + self.bias)


class SpikeActor(nn.Module):
    def __init__(self, obs_dim, hidden_sizes, act_dim):
        super().__init__()
        self.Linear1 = nn.Linear(obs_dim * 2 * encoder_pop_dim, hidden_sizes[0])
        self.Linear2 = nn.Linear(hidden_sizes[0], hidden_sizes[1])
        self.Linear3 = nn.Linear(hidden_sizes[1], hidden_sizes[2])
        self.encoder = Encoder(obs_dim, encoder_pop_dim, device)
        self.decoder = Decoder(act_dim, decoder_pop_dim, device)
        self.p1 = hidden_sizes[0]
        self.p2 = hidden_sizes[0] + hidden_sizes[1]
        for layer in (self.Linear1, self.Linear2, self.Linear3):
            nn.init.kaiming_normal_(layer.weight)
        self.record_spikes = False
        self.spk3_hist = []

    def forward(self, x, m):
        spk1, mem1 = lif_step(self.Linear1(self.encoder(x)), m[:, :self.p1])
        spk2, mem2 = lif_step(self.Linear2(spk1), m[:, self.p1:self.p2])
        spk3, mem3 = lif_step(self.Linear3(spk2), m[:, self.p2:])
        action_mean = self.decoder(spk3)
        mem_out = torch.cat([mem1, mem2, mem3], dim=1)
        if self.record_spikes:
            self.spk3_hist.append(spk3.detach().cpu().numpy())
        return action_mean, mem_out

    def reset_spike_records(self):
        self.spk3_hist = []


class SNNMultiCriticActorCritic(nn.Module):
    """1つのSNN Actor + reward groupごとの独立Value Critic。"""
    def __init__(self, obs_dim, critic_obs_dim, hidden_sizes, act_dim, num_critics=2):
        super().__init__()
        self.san = SpikeActor(obs_dim, hidden_sizes, act_dim)
        self.critics = nn.ModuleList([
            MLP([critic_obs_dim, hidden_sizes[0], hidden_sizes[1], 1]) for _ in range(num_critics)
        ])
        self.log_std = nn.Parameter(torch.full((act_dim,), float(np.log(PPO_INIT_STD)), device=device))

    def _dist(self, obs, mem):
        mean, mem2 = self.san(obs, mem)
        std = torch.exp(self.log_std).clamp(min=PPO_MIN_STD, max=PPO_MAX_STD).expand_as(mean)
        return torch.distributions.Normal(mean, std), mean, mem2

    @torch.no_grad()
    def act(self, obs, mem):
        dist, mean, mem2 = self._dist(obs, mem)
        action = dist.sample()
        logp = dist.log_prob(action).sum(-1)
        return action, logp, mean, dist.scale, mem2

    def evaluate_actions(self, obs, mem, actions):
        dist, mean, mem2 = self._dist(obs, mem)
        logp = dist.log_prob(actions).sum(-1)
        entropy = dist.entropy().sum(-1)
        return logp, entropy, mean, mem2

    def evaluate_critics(self, critic_obs):
        return torch.cat([critic(critic_obs) for critic in self.critics], dim=1)

    @torch.no_grad()
    def act_inference(self, obs, mem):
        mean, mem2 = self.san(obs, mem)
        return mean, mem2


class RolloutStorage:
    """APEX Multi-Critic PPO rollout buffer. Replay Bufferは使わない。"""
    def __init__(self, T, N, obs_dim, critic_obs_dim, act_dim, mem_dim, num_critics, device):
        z = lambda *s: torch.zeros(*s, dtype=torch.float32, device=device)
        self.obs = z(T, N, obs_dim)
        self.critic_obs = z(T, N, critic_obs_dim)
        self.mem = z(T, N, mem_dim)
        self.actions = z(T, N, act_dim)
        self.logp = z(T, N)
        self.mu = z(T, N, act_dim)
        self.sigma = z(T, N, act_dim)
        self.values = z(T, N, num_critics)
        self.next_values = z(T, N, num_critics)
        self.rewards = z(T, N, num_critics)
        self.dones = torch.zeros(T, N, dtype=torch.bool, device=device)
        self.terminals = torch.zeros(T, N, dtype=torch.bool, device=device)
        self.returns = z(T, N, num_critics)
        self.advantages = z(T, N)
        self.T, self.N, self.device = T, N, device
        self.step = 0

    def clear(self):
        self.step = 0

    @torch.no_grad()
    def add(self, obs, critic_obs, mem, actions, logp, mu, sigma, values, next_values,
            rewards, dones, terminals):
        t = self.step
        self.obs[t].copy_(obs); self.critic_obs[t].copy_(critic_obs)
        self.mem[t].copy_(mem); self.actions[t].copy_(actions)
        self.logp[t].copy_(logp); self.mu[t].copy_(mu); self.sigma[t].copy_(sigma)
        self.values[t].copy_(values)
        self.next_values[t].copy_(next_values); self.rewards[t].copy_(rewards)
        self.dones[t].copy_(dones); self.terminals[t].copy_(terminals)
        self.step += 1

    @torch.no_grad()
    def compute_returns(self, gamma, lam, critic_weights):
        assert self.step == self.T
        gae = torch.zeros(self.N, self.rewards.shape[-1], device=self.device)
        for t in reversed(range(self.T)):
            # timeoutはdeltaではbootstrapするが、GAEを次episodeへは伝播させない
            bootstrap_mask = (~self.terminals[t]).float().unsqueeze(1)
            continue_mask = (~self.dones[t]).float().unsqueeze(1)
            delta = self.rewards[t] + gamma * self.next_values[t] * bootstrap_mask - self.values[t]
            gae = delta + gamma * lam * continue_mask * gae
            self.returns[t] = gae + self.values[t]

        # APEX公式: reward groupごとにadvantageを正規化してから重み付き和
        adv_mc = torch.zeros(self.T, self.N, device=self.device)
        for g, w in enumerate(critic_weights):
            adv_g = self.returns[:, :, g] - self.values[:, :, g]
            adv_g = (adv_g - adv_g.mean()) / (adv_g.std() + 1e-8)
            adv_mc += float(w) * adv_g
        self.advantages.copy_(adv_mc)

    def minibatches(self, num_mini_batches, num_epochs):
        """非recurrent fallback。通常はsequence_minibatchesを使用する。"""
        B = self.T * self.N
        mb = B // num_mini_batches
        flat = lambda x: x.reshape(B, *x.shape[2:]) if x.dim() > 2 else x.reshape(B)
        obs, cobs, mem = flat(self.obs), flat(self.critic_obs), flat(self.mem)
        actions, old_logp = flat(self.actions), flat(self.logp)
        old_mu, old_sigma = flat(self.mu), flat(self.sigma)
        old_values, returns, adv = flat(self.values), flat(self.returns), flat(self.advantages)
        for _ in range(num_epochs):
            perm = torch.randperm(B, device=self.device)
            for i in range(num_mini_batches):
                idx = perm[i*mb:(i+1)*mb]
                yield obs[idx], cobs[idx], mem[idx], actions[idx], old_logp[idx], old_mu[idx], old_sigma[idx], \
                      old_values[idx], returns[idx], adv[idx]

    def sequence_minibatches(self, num_mini_batches, num_epochs):
        """SNN用: time軸を壊さず、環境軸だけをshuffleするtruncated recurrent PPO batch。"""
        if self.N % num_mini_batches != 0:
            raise ValueError("NUM_ENVS must be divisible by num_mini_batches for sequence PPO")
        envs_per_mb = self.N // num_mini_batches
        for _ in range(num_epochs):
            perm = torch.randperm(self.N, device=self.device)
            for i in range(num_mini_batches):
                idx = perm[i*envs_per_mb:(i+1)*envs_per_mb]
                yield (
                    self.obs[:, idx], self.critic_obs[:, idx], self.mem[0, idx],
                    self.actions[:, idx], self.logp[:, idx], self.mu[:, idx], self.sigma[:, idx],
                    self.values[:, idx], self.returns[:, idx], self.advantages[:, idx], self.dones[:, idx]
                )


def new_mem(n):
    return torch.zeros(n, mem_dim, device=device)


## 8. Multi-Critic PPO更新・評価ユーティリティ


In [10]:
def _check_finite(name, x):
    if not torch.isfinite(x).all():
        finite = x[torch.isfinite(x)]
        lo = float(finite.min()) if finite.numel() else float("nan")
        hi = float(finite.max()) if finite.numel() else float("nan")
        raise FloatingPointError(f"{name} contains NaN/Inf (finite range={lo:.4g}..{hi:.4g})")


def _evaluate_sequence(policy, obs_seq, init_mem, action_seq, done_seq):
    """現在のSNN policyでrollout系列を再生してlogp/mean/stdを再計算する。

    minibatchは環境軸だけshuffleし、time軸は保持する。episode終了後のmembraneは0へresetする。
    """
    mem = init_mem
    logps, ents, means, sigmas = [], [], [], []
    for t in range(obs_seq.shape[0]):
        logp_t, ent_t, mean_t, mem2 = policy.evaluate_actions(obs_seq[t], mem, action_seq[t])
        std_t = torch.exp(policy.log_std).clamp(min=PPO_MIN_STD, max=PPO_MAX_STD).expand_as(mean_t)
        logps.append(logp_t)
        ents.append(ent_t)
        means.append(mean_t)
        sigmas.append(std_t)
        # recurrent PPOと同様、次stepのstateだけdoneでresetする。
        mem = torch.where(done_seq[t].unsqueeze(1), torch.zeros_like(mem2), mem2)
    return (torch.stack(logps), torch.stack(ents),
            torch.stack(means), torch.stack(sigmas))


def ppo_update(policy, optimizer, storage):
    policy.train()
    policy_loss_sum = 0.0
    value_loss_sum = 0.0
    entropy_sum = 0.0
    kl_sum = 0.0
    grad_norm_sum = 0.0
    max_abs_log_ratio_seen = 0.0
    n_updates = 0

    if PPO_SNN_SEQUENCE_MINIBATCH:
        generator = storage.sequence_minibatches(num_mini_batches, ppo_epochs)
    else:
        generator = storage.minibatches(num_mini_batches, ppo_epochs)

    for batch in generator:
        if PPO_SNN_SEQUENCE_MINIBATCH:
            (obs_seq, cobs_seq, init_mem, act_seq, old_logp_seq, old_mu_seq, old_sigma_seq,
             old_values_seq, returns_seq, adv_seq, done_seq) = batch
            new_logp_seq, entropy_seq, mu_seq, sigma_seq = _evaluate_sequence(
                policy, obs_seq, init_mem, act_seq, done_seq
            )
            new_logp = new_logp_seq.reshape(-1)
            entropy = entropy_seq.reshape(-1)
            mu = mu_seq.reshape(-1, mu_seq.shape[-1])
            sigma = sigma_seq.reshape(-1, sigma_seq.shape[-1])
            old_logp_b = old_logp_seq.reshape(-1)
            old_mu_b = old_mu_seq.reshape(-1, old_mu_seq.shape[-1])
            old_sigma_b = old_sigma_seq.reshape(-1, old_sigma_seq.shape[-1])
            cobs_b = cobs_seq.reshape(-1, cobs_seq.shape[-1])
            old_values_b = old_values_seq.reshape(-1, old_values_seq.shape[-1])
            returns_b = returns_seq.reshape(-1, returns_seq.shape[-1])
            adv_b = adv_seq.reshape(-1)
        else:
            (obs_b, cobs_b, mem_b, act_b, old_logp_b, old_mu_b, old_sigma_b,
             old_values_b, returns_b, adv_b) = batch
            new_logp, entropy, mu, _ = policy.evaluate_actions(obs_b, mem_b, act_b)
            sigma = torch.exp(policy.log_std).clamp(min=PPO_MIN_STD, max=PPO_MAX_STD).expand_as(mu)

        for name, tensor in (("new_logp", new_logp), ("old_logp", old_logp_b),
                             ("advantage", adv_b), ("returns", returns_b),
                             ("mu", mu), ("sigma", sigma),
                             ("old_mu", old_mu_b), ("old_sigma", old_sigma_b)):
            _check_finite(name, tensor)

        # APEX公式と同じGaussian KLでadaptive learning rate。
        with torch.no_grad():
            kl = torch.sum(
                torch.log(sigma / old_sigma_b + 1.0e-5)
                + (old_sigma_b.square() + (old_mu_b - mu).square()) / (2.0 * sigma.square())
                - 0.5,
                dim=-1,
            )
            _check_finite("KL", kl)
            kl_mean = kl.mean()
            lr = float(optimizer.param_groups[0]["lr"])
            if kl_mean > PPO_DESIRED_KL * 2.0:
                lr = max(PPO_LR_MIN, lr / 1.5)
            elif 0.0 < kl_mean < PPO_DESIRED_KL / 2.0:
                lr = min(PPO_LR_MAX, lr * 1.5)
            for pg in optimizer.param_groups:
                pg["lr"] = lr

        log_ratio = new_logp - old_logp_b
        _check_finite("log_ratio", log_ratio)
        max_abs_log_ratio_seen = max(max_abs_log_ratio_seen, float(log_ratio.abs().max().detach()))
        # exp overflowだけを数値的に防ぐ。通常はadaptive KLが先にLRを下げる。
        ratio = torch.exp(torch.clamp(log_ratio, -PPO_LOG_RATIO_CLIP, PPO_LOG_RATIO_CLIP))
        unclipped = ratio * adv_b
        clipped = torch.clamp(ratio, 1.0 - PPO_CLIP, 1.0 + PPO_CLIP) * adv_b
        policy_loss = -torch.minimum(unclipped, clipped).mean()

        values = policy.evaluate_critics(cobs_b)
        _check_finite("critic values", values)
        value_clipped = old_values_b + (values - old_values_b).clamp(-PPO_CLIP, PPO_CLIP)
        value_loss_unclipped = (values - returns_b).pow(2)
        value_loss_clipped = (value_clipped - returns_b).pow(2)
        value_loss = torch.maximum(value_loss_unclipped, value_loss_clipped).mean()

        ent = entropy.mean()
        loss = policy_loss + PPO_VALUE_COEF * value_loss - PPO_ENTROPY_COEF * ent
        _check_finite("PPO loss", loss.reshape(1))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(
            policy.parameters(), PPO_MAX_GRAD_NORM, error_if_nonfinite=True
        )
        optimizer.step()

        with torch.no_grad():
            _check_finite("log_std after optimizer.step", policy.log_std)
            policy.log_std.clamp_(
                min=float(np.log(PPO_MIN_STD)),
                max=float(np.log(PPO_MAX_STD)),
            )

        policy_loss_sum += float(policy_loss.detach())
        value_loss_sum += float(value_loss.detach())
        entropy_sum += float(ent.detach())
        kl_sum += float(kl_mean.detach())
        grad_norm_sum += float(grad_norm.detach())
        n_updates += 1

    return {
        "policy_loss": policy_loss_sum / max(n_updates, 1),
        "value_loss": value_loss_sum / max(n_updates, 1),
        "entropy": entropy_sum / max(n_updates, 1),
        "action_std": float(torch.exp(policy.log_std).clamp(max=PPO_MAX_STD).mean().detach()),
        "kl": kl_sum / max(n_updates, 1),
        "lr": float(optimizer.param_groups[0]["lr"]),
        "grad_norm": grad_norm_sum / max(n_updates, 1),
        "max_abs_log_ratio": max_abs_log_ratio_seen,
    }


@torch.no_grad()
def refresh_snn_memory_after_update(policy, storage):
    """PPO更新後、直前rolloutを新policyで再生して次rollout用membrane stateを更新する。

    rollout開始時のhidden stateはstandard recurrent PPOと同じく保存値を使い、
    その後の32 stepは更新後policyで再構築する。これによりoptimizer.step後に古いmemをそのまま持ち越さない。
    """
    policy.eval()
    mem = storage.mem[0].clone()
    for t in range(storage.T):
        _, mem2 = policy.san(storage.obs[t], mem)
        mem = torch.where(storage.dones[t].unsqueeze(1), torch.zeros_like(mem2), mem2)
    return mem


def decap_factor(control_step):
    """DecAP。4096-env公式runと同程度のtransition量になるようAPEX_KをNUM_ENVSで補正。"""
    return max(APEX_PRIOR_MIN, min(1.0, APEX_GAMMA ** (float(control_step) / APEX_K)))


TEST_CMDS = {
    "前進":  np.array([0.60, 0.0, 0.0]),
    "後退":  np.array([-0.60, 0.0, 0.0]),
    "左移動": np.array([0.0, 0.60, 0.0]),
    "左旋回": np.array([0.0, 0.0, 1.00]),
}
TEST_CMD_MAT = np.stack(list(TEST_CMDS.values()))

VIDEO_TASKS = list(TEST_CMDS)                      # 動画の 2x2 配置順 (左上→右上→左下→右下)
VIDEO_LABELS = {"前進": "forward", "後退": "backward", "左移動": "left", "左旋回": "turn left"}
VIDEO_WORLDS = [list(TEST_CMDS).index(n) for n in VIDEO_TASKS]
_VIDEO_FONT = ImageFont.load_default(size=22)      # 既定フォントは日本語が出ないのでラベルはローマ字


def grid_frame(env, fallen=None):
    """4 タスクのワールドを描画し、タスク名と指令を焼き込んで 2x2 に並べた 1 フレームを返す。

    fallen[N] (bool) を渡すと、転倒したワールドにだけ FALLEN を表示する。
    時間切れ (max_ep_len 到達) は転倒ではないので表示しない。
    """
    panels = []
    for name, w in zip(VIDEO_TASKS, VIDEO_WORLDS):
        img = Image.fromarray(env.render(world=w))
        cmd = TEST_CMD_MAT[w]
        down = False if fallen is None else bool(fallen[w])
        txt = (f"{VIDEO_LABELS[name]}  [{cmd[0]:+.2f} {cmd[1]:+.2f} {cmd[2]:+.2f}]"
               + ("  FALLEN" if down else ""))
        ImageDraw.Draw(img).text((10, 8), txt, font=_VIDEO_FONT, fill=(255, 255, 255),
                                 stroke_width=2, stroke_fill=(0, 0, 0))
        panels.append(np.asarray(img))
    return np.concatenate([np.concatenate(panels[:2], axis=1),
                           np.concatenate(panels[2:], axis=1)], axis=0)



def test_agent(video_path=None):
    """評価ではAction Priorを完全にOFFにし、SNN Actor単体を評価する。"""
    policy.eval()
    o = test_env.reset(command=TEST_CMD_MAT)
    m = new_mem(test_env.N)
    ep_ret = torch.zeros(test_env.N, device=device)
    alive = torch.ones(test_env.N, dtype=torch.bool, device=device)
    fell = torch.zeros(test_env.N, dtype=torch.bool, device=device)
    writer_v = imageio.get_writer(video_path, fps=50) if video_path else None
    n_frames = 0
    try:
        for _ in range(test_env.max_steps):
            on = obs_rms.normalize(o)
            with torch.no_grad():
                a, m2 = policy.act_inference(on, m)
            o2, rg, d, info = test_env.step(a, prior_factor=0.0)
            ep_ret += info["total_reward"] * alive
            fell |= info["fallen"]
            alive &= ~d
            o = o2
            m = torch.where(d.unsqueeze(1), new_mem(test_env.N), m2)
            if writer_v is not None:
                writer_v.append_data(grid_frame(test_env, fell))
                n_frames += 1
            if not bool(alive.any()):
                break
    finally:
        if writer_v is not None:
            writer_v.close()
    rets = {name: round(v, 1) for name, v in zip(TEST_CMDS, ep_ret.tolist())}
    return (rets, n_frames) if video_path else rets


def save_training_video(it):
    path = f"{video_dir}/{RUN_NAME}_{it}it_4tasks.mp4"
    tmp = os.path.join(tempfile.gettempdir(), os.path.basename(path))
    try:
        rets, n_frames = test_agent(video_path=tmp)
        shutil.move(tmp, path)
    except Exception as e:
        print(f"  動画保存に失敗 (学習は継続): {type(e).__name__}: {e}")
        traceback.print_exc(limit=3)
        return
    scores = " / ".join(f"{n}={rets[n]}" for n in VIDEO_TASKS)
    print(f"  動画保存: {path} ({n_frames}フレーム, 報酬 {scores})")


## 9. 環境・SNN Multi-Critic Policyの生成


In [11]:
# 学習用：rough terrainならAPEX terrain curriculumを使用。
EnvClass = Go2ApexTerrainWarpEnv if USE_APEX_ROUGH_TERRAIN else Go2ImitationWarpEnv

env = EnvClass(
    NUM_ENVS, device, xml_path=TRAIN_XML,
    max_ep_len=max_ep_len, resample_cmd=True, seed=0,
    terrain_meta=APEX_TERRAIN_META, terrain_curriculum=USE_APEX_ROUGH_TERRAIN,
) if USE_APEX_ROUGH_TERRAIN else EnvClass(
    NUM_ENVS, device, xml_path=TRAIN_XML,
    max_ep_len=max_ep_len, resample_cmd=True, seed=0,
)

# 評価は既定でflat + Action Prior OFF。TEST_ON_ROUGH_TERRAIN=Trueならroughで評価。
if TEST_ON_ROUGH_TERRAIN and USE_APEX_ROUGH_TERRAIN:
    test_env = Go2ApexTerrainWarpEnv(
        len(TEST_CMDS), device, xml_path=TEST_XML,
        max_ep_len=max_ep_len, resample_cmd=False, seed=1,
        terrain_meta=APEX_TERRAIN_META, terrain_curriculum=True,
    )
else:
    test_env = Go2ImitationWarpEnv(
        len(TEST_CMDS), device, xml_path=TEST_XML,
        max_ep_len=max_ep_len, resample_cmd=False, seed=1,
    )

seed = 0
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

obs_dim = env.obs_dim
critic_obs_dim = env.critic_obs_dim
act_dim = env.act_dim
hidden_sizes = (hidden[0], hidden[1], act_dim * 2 * decoder_pop_dim)
mem_dim = sum(hidden_sizes)

policy = SNNMultiCriticActorCritic(
    obs_dim=obs_dim,
    critic_obs_dim=critic_obs_dim,
    hidden_sizes=hidden_sizes,
    act_dim=act_dim,
    num_critics=NUM_CRITICS,
).to(device)

if SNN_COMPILE:
    policy.san.forward = torch.compile(policy.san.forward, dynamic=False)
    print("SNN actorをtorch.compileしました")

optimizer = torch.optim.Adam(policy.parameters(), lr=PPO_LR, fused=True)
storage = RolloutStorage(num_steps_per_env, NUM_ENVS, obs_dim, critic_obs_dim,
                         act_dim, mem_dim, NUM_CRITICS, device)
obs_rms = RunningMeanStd(obs_dim, device, NORM_CLIP_LIMIT)
critic_rms = RunningMeanStd(critic_obs_dim, device, NORM_CLIP_LIMIT)

print(f"TRAIN_XML={TRAIN_XML}")
print(f"obs_dim={obs_dim} / critic_obs_dim={critic_obs_dim} / act_dim={act_dim} / mem_dim={mem_dim}")
if USE_APEX_ROUGH_TERRAIN:
    print(f"terrain: {TERRAIN_ROWS} levels x {TERRAIN_COLS} cols / initial <= {TERRAIN_MAX_INIT_LEVEL}")
print(policy)

test_env.render(world=0)
atexit.register(env.close)
atexit.register(test_env.close)


torque limits [Nm]: [23.7, 23.7, 45.43, 23.7, 23.7, 45.43, 23.7, 23.7, 45.43, 23.7, 23.7, 45.43]
APEX contact bodies: ['FL_foot', 'FR_foot', 'RL_foot', 'RR_foot']
torque limits [Nm]: [23.7, 23.7, 45.43, 23.7, 23.7, 45.43, 23.7, 23.7, 45.43, 23.7, 23.7, 45.43]
APEX contact bodies: ['FL_foot', 'FR_foot', 'RL_foot', 'RR_foot']
TRAIN_XML=/workspace/easy-docker-jupyterhub/unitree_mujoco/unitree_robots/go2_2/go2_snn_apex_rough_fixed1024_apex_curriculum.xml
obs_dim=45 / critic_obs_dim=264 / act_dim=12 / mem_dim=6656
terrain: 10 levels x 20 cols / initial <= 5
SNNMultiCriticActorCritic(
  (san): SpikeActor(
    (Linear1): Linear(in_features=5760, out_features=256, bias=True)
    (Linear2): Linear(in_features=256, out_features=256, bias=True)
    (Linear3): Linear(in_features=256, out_features=6144, bias=True)
    (encoder): Encoder(
      (activation): Tanh()
    )
    (decoder): Decoder(
      (activation): Tanh()
    )
  )
  (critics): ModuleList(
    (0-1): 2 x MLP(
      (Linear1): Linear(

<bound method Go2ImitationWarpEnv.close of <__main__.Go2ImitationWarpEnv object at 0x7fe48c490490>>

## 10. APEX Multi-Critic PPO 学習ループ

1 iterationごとに on-policy rollout を収集し、2つのCriticでGAEを計算します。

- `reward[:,0]`: APEX Style = joint-angle imitation + foot-position imitation
- `reward[:,1]`: APEX Task = velocity tracking + regularization
- 各Advantageを**別々に正規化**し、`0.5 * A_style + 0.5 * A_task`としてActorを更新
- Action Priorは `c_n = 0.99^(control_step/100)` で減衰
- 評価では `c_n = 0`


In [ ]:
writer = SummaryWriter(logdir)

o = env.reset()
c = env.get_critic_obs()
obs_rms.update(o)
critic_rms.update(c)
m = new_mem(NUM_ENVS)

ep_ret = torch.zeros(NUM_ENVS, device=device)
ep_style = torch.zeros(NUM_ENVS, device=device)
ep_task = torch.zeros(NUM_ENVS, device=device)
ep_len = torch.zeros(NUM_ENVS, device=device)
ret_sum = torch.zeros((), device=device)
style_sum = torch.zeros((), device=device)
task_sum = torch.zeros((), device=device)
len_sum = torch.zeros((), device=device)
ret_cnt = torch.zeros((), device=device)
parts_sum = torch.zeros(len(Go2ImitationWarpEnv.REWARD_KEYS), device=device)
fallen_sum = torch.zeros((), device=device)
n_parts = 0
t0 = time.time()
control_steps = 0  # vectorized env全体で共有する環境control-step counter（NUM_ENVSは掛けない）

for it in range(total_iters):
    storage.clear()
    prior_sum = 0.0
    prior_last = decap_factor(control_steps)
    norm_obs_samples = []
    norm_critic_samples = []

    # rollout中はRMSを固定。PPOに保存したnormalized obsとold log-probを完全に対応させる。
    for t in range(num_steps_per_env):
        on = obs_rms.normalize(o)
        cn = critic_rms.normalize(c)

        with torch.no_grad():
            a, logp, mu, sigma, m2 = policy.act(on, m)
            values = policy.evaluate_critics(cn)

        # APEX公式と同様に、Action PriorはPPO iterationではなくcontrol stepごとに減衰
        pf = decap_factor(control_steps)
        prior_sum += pf
        prior_last = pf
        o2, reward_groups, d, info = env.step(a, prior_factor=pf)
        control_steps += 1
        c2 = env.get_critic_obs()             # reset前terminal stateのcritic obs
        with torch.no_grad():
            c2n = critic_rms.normalize(c2)
            next_values = policy.evaluate_critics(c2n)

        storage.add(on, cn, m, a, logp, mu, sigma, values, next_values,
                    reward_groups, d, info["fallen"])

        ep_ret += info["total_reward"]
        ep_style += reward_groups[:, 0]
        ep_task += reward_groups[:, 1]
        ep_len += 1
        parts_sum += info["reward_parts"].mean(0)
        fallen_sum += info["fallen"].float().mean()
        n_parts += 1

        done_f = d.float()
        ret_sum += (ep_ret * done_f).sum()
        style_sum += (ep_style * done_f).sum()
        task_sum += (ep_task * done_f).sum()
        len_sum += (ep_len * done_f).sum()
        ret_cnt += d.sum()
        ep_ret *= ~d; ep_style *= ~d; ep_task *= ~d; ep_len *= ~d

        # reset後のstateを次stepのcurrent stateへ
        o = env.autoreset(d)
        c = env.get_critic_obs()
        m = torch.where(d.unsqueeze(1), new_mem(NUM_ENVS), m2)
        norm_obs_samples.append(o.detach())
        norm_critic_samples.append(c.detach())

    # APEX Multi-Critic GAE
    storage.compute_returns(PPO_GAMMA, PPO_LAMBDA, CRITIC_WEIGHTS)
    metrics = ppo_update(policy, optimizer, storage)
    # optimizer.stepでSNN重みが変わった後に、古い膜電位をそのまま次rolloutへ持ち越さない。
    m = refresh_snn_memory_after_update(policy, storage)

    # 次のrolloutから新しいrunning statisticsを使う
    with torch.no_grad():
        obs_rms.update(torch.cat(norm_obs_samples, dim=0))
        critic_rms.update(torch.cat(norm_critic_samples, dim=0))

    steps = (it + 1) * transitions_per_iter

    # ---- checkpoint ----
    if (it + 1) % save_interval == 0 or (it + 1) == total_iters:
        tag = f"{steps//1000}Kit"
        # 実機推論で使うSNN Actor単体
        torch.save(policy.san.state_dict(), f"{model_dir}/{RUN_NAME}_actor_{tag}.pt")
        # 学習再開用full checkpoint
        torch.save({
            "iteration": it + 1,
            "steps": steps,
            "control_steps": control_steps,
            "policy": policy.state_dict(),
            "optimizer": optimizer.state_dict(),
            "obs_rms": obs_rms.state_dict(),
            "critic_rms": critic_rms.state_dict(),
            "config": {
                "use_rsi": USE_RSI,
                "actor_use_phase": False,
                "actor_obs": "APEX_45D",
                "style_reward": "joint_angle+foot_position+quaternion",
                "task_reward": "tracking+torque+dof_acc+collision+action_rate+feet_slip+impact+stumble",
                "command_scale": CMD_SCALE.tolist(),
                "num_envs": NUM_ENVS,
                "servo_kp": SERVO_KP,
                "servo_kd": SERVO_KD,
                "action_scale": ACT_SCALE.tolist(),
                "desired_kl": PPO_DESIRED_KL,
                "snn_sequence_minibatch": PPO_SNN_SEQUENCE_MINIBATCH,
                "apex_gamma": APEX_GAMMA,
                "apex_k": APEX_K,
                "critic_weights": CRITIC_WEIGHTS,
                "rough_terrain": USE_APEX_ROUGH_TERRAIN,
                "terrain_rows": TERRAIN_ROWS,
                "terrain_cols": TERRAIN_COLS,
                "terrain_seed": TERRAIN_SEED,
            },
        }, f"{model_dir}/{RUN_NAME}_full_{tag}.pt")
        np.savez(f"{model_dir}/{RUN_NAME}_norm_{tag}.npz",
                 mean=obs_rms.mean.detach().cpu().numpy(),
                 var=obs_rms.var.detach().cpu().numpy())
        save_training_video(it + 1)

    # ---- evaluation + log ----
    if (it + 1) % eval_interval == 0 or it == 0:
        res = test_agent()
        n_ep = ret_cnt.item()
        avg_ret = ret_sum.item() / n_ep if n_ep else float('nan')
        avg_style = style_sum.item() / n_ep if n_ep else float('nan')
        avg_task = task_sum.item() / n_ep if n_ep else float('nan')
        avg_len = len_sum.item() / n_ep if n_ep else float('nan')
        if n_ep:
            ret_sum.zero_(); style_sum.zero_(); task_sum.zero_(); len_sum.zero_(); ret_cnt.zero_()

        writer.add_scalar("train/episode_total", avg_ret, steps)
        writer.add_scalar("train/episode_style", avg_style, steps)
        writer.add_scalar("train/episode_task", avg_task, steps)
        writer.add_scalar("train/episode_length", avg_len, steps)
        prior_mean = prior_sum / num_steps_per_env
        writer.add_scalar("train/decap_factor_mean", prior_mean, steps)
        writer.add_scalar("train/decap_factor_last", prior_last, steps)
        if USE_APEX_ROUGH_TERRAIN and hasattr(env, "terrain_levels"):
            writer.add_scalar("terrain/mean_level", env.terrain_levels.float().mean().item(), steps)
            for col_group, name in enumerate(Go2ApexTerrainWarpEnv.TERRAIN_TYPES):
                # 20 columns / 5種類なので4列ずつ集約
                lo = col_group * (TERRAIN_COLS // 5)
                hi = (col_group + 1) * (TERRAIN_COLS // 5)
                mask = (env.terrain_types >= lo) & (env.terrain_types < hi)
                if bool(mask.any()):
                    writer.add_scalar(f"terrain/{name}_level", env.terrain_levels[mask].float().mean().item(), steps)
        writer.add_scalar("loss/policy", metrics["policy_loss"], steps)
        writer.add_scalar("loss/value", metrics["value_loss"], steps)
        writer.add_scalar("policy/entropy", metrics["entropy"], steps)
        writer.add_scalar("policy/action_std", metrics["action_std"], steps)
        writer.add_scalar("policy/kl", metrics["kl"], steps)
        writer.add_scalar("policy/lr", metrics["lr"], steps)
        writer.add_scalar("policy/grad_norm", metrics["grad_norm"], steps)
        writer.add_scalar("policy/max_abs_log_ratio", metrics["max_abs_log_ratio"], steps)
        for name, v in res.items():
            writer.add_scalar(f"test/{name}", v, steps)
        if n_parts:
            for tag, v in zip(Go2ImitationWarpEnv.REWARD_KEYS, (parts_sum / n_parts).tolist()):
                writer.add_scalar(tag, v, steps)
            writer.add_scalar("train/fallen_rate", fallen_sum.item() / n_parts, steps)
            parts_sum.zero_(); fallen_sum.zero_(); n_parts = 0

        print(f"{it+1:5d}/{total_iters} it  {steps/1e6:6.2f}M transitions  "
              f"prior={prior_last:5.3f}  ret={avg_ret:7.2f}  style={avg_style:7.2f} task={avg_task:7.2f}  "
              f"PPO(policy={metrics['policy_loss']:+.4f}, value={metrics['value_loss']:.4f}, "
              f"std={metrics['action_std']:.3f}, kl={metrics['kl']:.4f}, lr={metrics['lr']:.1e})  "
              f"test={res}  ({time.time()-t0:.0f}s)")

writer.close()
print("学習完了")

if ENV_COLAB:
    runtime.unassign()


    1/2000 it    0.03M transitions  prior=0.999  ret=    nan  style=    nan task=    nan  PPO(policy=+0.0025, value=324.1454, std=0.301, kl=0.0108, lr=4.5e-04)  test={'前進': -14.9, '後退': -14.2, '左移動': -14.9, '左旋回': -14.3}  (9s)


In [ ]:
# 学習後の重要事項
# 1) 評価・実機では Decaying Action Prior を使わない (prior_factor=0)。
# 2) 実機推論に必要なのは *_actor_*.pt と *_norm_*.npz。
# 3) 不整地学習に進む場合は TRAIN_XML を不整地sceneへ変更し、まず他の条件を固定して比較する。
# 4) APEXの厳密なablationを行う場合は USE_RSI / ACTOR_USE_PHASE / CRITIC_WEIGHTS を切り替える。


## 11. 実装上の対応関係と注意点

このNotebookは **APEX公式コードそのもののコピーではなく、MuJoCo-Warp + SNNへの移植版**です。今回の版では、公式との差分のうち学習安定性に直接関係する箇所を優先して修正しています。

### 今回修正した重要点

- **1024 environments** で学習
- command range: **vx = ±0.6 m/s, vy = ±0.6 m/s, wz = ±1.0 rad/s**
- APEX Go2 controlに合わせて **Kp=20 / Kd=0.5 / action scale=0.25**
- Actor action clipをAPEX同様100へ緩和し、代わりに **最終torqueをMJCF/URDF limitでclip**
- Action PriorとStyle rewardが同じreference indexを使うよう1-stepずれを修正
- 1024 envでは、4096-env公式runと同程度のtransition数をPrior付きで経験できるよう **DecAP k=400**（smoke testではenv数に応じ自動補正）
- privileged criticを **APEX 77D相当**へ変更（phase / absolute ref joint / ref foot / ref quaternionを追加）
- Style: joint-angle + foot-position + quaternion imitation
- Task: tracking + torque + DOF acceleration + collision + action-rate + feet-slip + impact-reduction + rough時stumble/ang-vel-xy
- APEX公式PPOと同様にrolloutの **old mean / old sigma** を保存し **Adaptive KL (desired KL=0.01)** を追加
- `NaN/Inf`、gradient非有限、log-ratio異常を早期検出
- SNNはtime軸をランダムshuffleせず、環境軸だけをshuffleする **sequence minibatch** で膜電位を順に再計算
- PPO更新後も直前rolloutを新policyで再生し、次rollout用membrane stateを更新
- rough criticのheight observationにAPEXと同じscale 5.0を適用
- base/hip接触によるterminationを追加

### 77D Criticのreference quaternionについて

公式APEXはモーションデータにbase quaternionを持ちますが、このNotebookの参照軌道は手続き生成トロットでbase quaternion trajectoryを持ちません。そのため、reference quaternionには **upright identity quaternion `[1,0,0,0]` (MuJoCo wxyz)** を使用しています。ここは公式データをそのまま再生する場合との残る差分です。

### MuJoCo-Warp移植上の注意

- Isaac Gymの `net_contact_force_tensor` の代わりにMuJoCo-Warpの `cfrc_ext` を使用します。
- `self.m.opt.run_rne_postconstraint = True` を有効化し、collision/stumble/termination用の外力を更新します。
- MJCFに独立したfoot bodyがない場合はcalf bodyの外力を足接触として使います。
- torque limitはMJCFの有効な `actuator_forcerange` / `ctrlrange` を優先し、無い場合のみGo2 URDF相当値へfallbackします。


## 12. 保存先と地形データ

### 保存フォルダ

日付が09-24、`RUN_NAME="go2_snn_apex_rough"` の例：

```text
params_09-24_APEX/
  go2_snn_apex_rough_actor_*.pt
  go2_snn_apex_rough_full_*.pt
  go2_snn_apex_rough_norm_*.npz

videos_09-24_APEX/
  go2_snn_apex_rough_*it_4tasks.mp4

terrains_09-24_APEX/
  go2_snn_apex_rough_terrain_atlas.xml
  go2_snn_apex_rough_terrain_meta.npz
```

`RUN_NAME`を変更すれば保存される各ファイルのprefixも変更できます。

### Terrain XML

APEX公式はheight fieldを実行時生成してIsaac Gymのtrimeshへ変換します。本Notebookでは同じterrain curriculumをMuJoCoへ移植し、height fieldをMJCF `<hfield>` としてXML保存します。Actorはterrain heightを直接見ず、Criticだけが187点のheight informationをprivileged observationとして使用します。
